<p class="lead">Supervised learning asks a question with a known answer. Clustering asks the data what groups it contains, and the data always answers — even when there are no groups. You will meet the five families of clustering algorithm the laboratory uses (centroid, hierarchical, density-based, model-based and the chemistry-specific leader algorithms), the distance functions that make them meaningful, and the validity machinery that decides whether a partition deserves to be in a paper. Four lab tasks run through the unit: detect the chemical series inside the ROCK1 dataset, pick a diverse subset and <em>prove</em> it is more diverse than a random one, extract metastable conformations from a replica molecular-dynamics ensemble, and ask whether the gingival expression slice contains patient subgroups. It ends with the error unsupervised learning invites most often: reading clusters off a t-SNE picture of data that has none.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata el aprendizaje no supervisado. Se estudian k-medias (con su geometría, su costo y sus modos de falla), el agrupamiento jerárquico aglomerativo con sus distintos enlaces y el dendrograma, los métodos basados en densidad (DBSCAN y OPTICS, implementados a mano porque el paquete <code>dbscan</code> no está instalado), las mezclas de gaussianas ajustadas con un algoritmo EM escrito a mano y seleccionadas por BIC, y el algoritmo de Butina sobre huellas moleculares. Se eligen métricas de distancia acordes al tipo de dato: euclidiana sobre descriptores estandarizados, Tanimoto sobre huellas binarias (escrita en álgebra de matrices dispersas), RMSD sobre conformaciones y correlación sobre perfiles de expresión. Se validan las particiones con índices internos (silueta, Davies-Bouldin, Calinski-Harabasz), externos (índice de Rand ajustado) y con estabilidad bajo remuestreo, mostrando los límites del método del codo. Finalmente se resuelven cuatro tareas del laboratorio - detección de series químicas, selección de un subconjunto diverso, conformaciones metaestables de dinámica molecular y subgrupos de pacientes - y se advierte sobre los grupos que no existen en los mapas t-SNE.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Apply k-means, hierarchical (agglomerative), density-based, Gaussian-mixture and Butina/leader clustering, and choose among them from the data's geometry and from the question being asked.</li>
<li>Choose distance metrics appropriate to the data type: Euclidean on standardised descriptors, Tanimoto/Jaccard on fingerprints, RMSD on conformations, correlation on expression profiles.</li>
<li>Validate a clustering with internal indices (silhouette, Davies-Bouldin, Calinski-Harabasz), external indices (adjusted Rand index) and stability under resampling, and resist over-interpretation.</li>
<li>Use clustering for the laboratory's practical tasks: diversity selection, series identification, representative conformations and patient subgroups.</li>
<li>Explain why an elbow appears in data that has no clusters, and why clusters appear in a t-SNE map of pure noise.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running; every number in the prose is printed by the chunk above it. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","cluster","Rtsne","irlba","patchwork"))</code> once before the first chunk. This is the first unit of Block D that uses no target variable.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(cluster)
  library(ggplot2);   library(patchwork); library(irlba); library(Rtsne)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| Matrix",     as.character(packageVersion("Matrix")),
    "| cluster",    as.character(packageVersion("cluster")),
    "| ggplot2",    as.character(packageVersion("ggplot2")),
    "| Rtsne",      as.character(packageVersion("Rtsne")), "\n")



The versions are printed so that a reader who gets a different number knows where to look first.



In [ ]:
dp <- function(f) file.path("..", "data", f)      # knitr runs with cwd = this unit's folder

df  <- fread(dp("rock1_chembl_curated.csv"))
cmp <- df[censored == FALSE]                       # exact-value compounds only
DESC <- c("MolWt","MolLogP","TPSA","HBD","HBA","RotB",
          "AromaticRings","RingCount","HeavyAtoms","FractionCSP3","QED")
Zdesc <- scale(as.matrix(cmp[, ..DESC]))           # 11 standardised descriptors

sp   <- fread(dp("rock1_morgan_sparse.csv.gz"))    # frozen Morgan bits: (InChIKey, bit)
keep <- sp[id %in% cmp$inchikey]
FP   <- sparseMatrix(i = match(keep$id, cmp$inchikey), j = keep$bit + 1L, x = 1,
                     dims = c(nrow(cmp), 2048))
cat(sprintf("compounds %d | descriptors %d | fingerprint bits %d | bits set per molecule %.1f\n",
            nrow(cmp), ncol(Zdesc), ncol(FP), mean(rowSums(FP))))



<div class="box bridge"><h4>Substitution — where the fingerprints come from</h4>
<p>R has no RDKit binding in this environment, so the Morgan fingerprints (radius 2, 2,048 bits) are <strong>not recomputed here</strong>: they are read from the frozen sparse file <code>data/rock1_morgan_sparse.csv.gz</code>, two columns <code>id</code> (InChIKey) and <code>bit</code> (0-based index of a set bit), and rebuilt as a sparse 0/1 matrix with <code>Matrix::sparseMatrix()</code>. The file was generated with RDKit 2026.03.4 exactly as the Python edition computes it in-notebook; <code>data/manifest_fingerprints.json</code> records the generator call and the SHA-256 of each file, and <code>data/rock1_curated.sdf.gz</code> ships the structures so that a reader with ChemmineR or rcdk can recompute the bits and check them. The 2,764 exact-value compounds carry about 52 set bits each out of 2,048 — a density of 2.6 %, which is why every similarity computation below is written in sparse-matrix algebra.</p></div>



In [ ]:
read_npy <- function(path) {                        # minimal .npy reader (no reticulate needed)
  con <- file(path, "rb"); on.exit(close(con)); readBin(con, "raw", 8)
  hlen <- readBin(con, "integer", 1, size = 2, endian = "little")
  h    <- rawToChar(readBin(con, "raw", hlen))
  descr <- sub(".*'descr': *'([^']+)'.*", "\\1", h)
  dims  <- as.integer(Filter(nzchar, strsplit(
             gsub("[ )]", "", sub(".*'shape': *\\(([^)]*)\\).*", "\\1", h)), ",")[[1]]))
  x <- readBin(con, if (substring(descr, 2, 2) == "f") "double" else "integer",
               prod(dims), size = as.integer(substring(descr, 3)), endian = "little")
  if (length(dims) > 1) aperm(array(x, dim = rev(dims)), rev(seq_along(dims))) else x
}
nd <- file.path(tempdir(), "npz"); dir.create(nd, showWarnings = FALSE)
unzip(dp("md_trpcage_ca_coords.npz"), exdir = nd)
RUNS <- c("A1","A2","A3","B1","B2","B3")
Fmd  <- do.call(rbind, lapply(RUNS, function(r) {    # 3000 frames x 60 coordinates (nm)
  a <- read_npy(file.path(nd, paste0(r, ".npy"))); t(apply(a, 1, function(m) as.vector(t(m)))) }))
ref_ca <- read_npy(file.path(nd, "ref_ca.npy")); time_ps <- read_npy(file.path(nd, "time_ps.npy"))
run_of_frame <- rep(RUNS, each = 500); time_of_frame <- rep(time_ps, 6)

ex <- fread(dp("geo_periodontitis_expr.csv")); gn <- ex[[1]]
E  <- as.matrix(ex[, -1]); rownames(E) <- gn
meta <- fread(dp("geo_periodontitis_meta.csv"))
cat(sprintf("MD: %d frames = %d runs x %d, %d C-alpha atoms, %d coordinates per frame\n",
            nrow(Fmd), length(RUNS), nrow(Fmd)/length(RUNS), ncol(Fmd)/3, ncol(Fmd)))
cat(sprintf("expression: %d genes x %d samples (%d diseased, %d healthy)\n", nrow(E), ncol(E),
            sum(meta$status == "diseased"), sum(meta$status == "healthy")))



Three data sets, three geometries: 2,764 compounds described twice, 3,000 molecular-dynamics frames of 60 coordinates each, and 138 biopsies of 5,000 genes. The `.npz` archive of superposed C-alpha coordinates is a zip of NumPy arrays, which base R reads with `unzip()` plus the twelve-line binary reader above — no Python interpreter is involved.

## 1. What clustering is, and what it is not

A **clustering algorithm** takes \(n\) objects and returns a label for each one without ever seeing a target variable. Everything that makes supervised learning self-correcting — a held-out score, a learning curve, a y-scrambling control — has to be rebuilt by hand here, and most published clusterings do not bother; Jain's survey of fifty years of the field opens with exactly that complaint<sup class="c"><a href="#r1">[1]</a></sup>.

Three distinctions organise the whole unit.

1. **The question comes before the algorithm.** "Which compounds belong to the same series?" is answered with Tanimoto distances and a leader algorithm; "which 50 compounds should we buy?" is a *coverage* question answered by a diversity picker, not by a clustering at all.
2. **The metric is a modelling decision, not a detail.** Two molecules are "close" in descriptor space, in fingerprint space and in scaffold space in three different and only weakly related senses, as Section 8 measures explicitly.
3. **A partition is a hypothesis.** It must be checked internally (are the clusters compact and separated?), externally (do they agree with something known?) and for stability (does a different sample give the same partition?).

<div class="box bridge"><h4>Domain context — what a "cluster" means in each of our data sets</h4>
<p><strong>Compounds.</strong> Medicinal chemists work in <em>series</em>: one core structure (a <strong>scaffold</strong>, formally the Bemis-Murcko framework — the molecule stripped of its side chains<sup class="c"><a href="#r20">[20]</a></sup>) decorated with dozens of substituent variations, so a cluster of compounds should ideally recover a series. A <strong>Morgan fingerprint</strong><sup class="c"><a href="#r19">[19]</a></sup> encodes a molecule as a bit vector, each bit recording one circular atom environment up to radius 2 hashed into 2,048 positions.</p>
<p><strong>Conformations.</strong> A molecular-dynamics (MD) simulation integrates Newton's equations for a protein in water and writes a <em>frame</em> of coordinates every few picoseconds. Our frozen ensemble is the 20-residue Trp-cage miniprotein (PDB 1L2Y), six independent 1 ns runs: three at 300 K (condition A), three at 340 K (condition B). The frames are superposed on a common reference, so the Euclidean distance between two C-alpha coordinate vectors divided by \(\sqrt{N}\) <em>is</em> their root-mean-square deviation (RMSD) in nanometres. A cluster of frames is a candidate <strong>metastable conformation</strong>.</p>
<p><strong>Patients.</strong> The expression slice holds log2 microarray intensities for 5,000 genes in 138 gingival-tissue biopsies from 69 patients, one diseased and one healthy site each<sup class="c"><a href="#r27">[27]</a></sup>. A cluster of samples is a candidate <strong>molecular subtype</strong>, a claim made for this very cohort<sup class="c"><a href="#r26">[26]</a></sup>.</p></div>

## 2. Distance first: four metrics for four kinds of data

Every clustering algorithm in this unit is a rule for grouping objects that are *close*. Fix the notion of closeness and you have already decided most of the result.

**Euclidean distance on standardised features.** \(d(\mathbf{x},\mathbf{y}) = \sqrt{\sum_j (x_j-y_j)^2}\). The sum runs over columns, so molecular weight in daltons (hundreds) would swamp logP in log units (single digits): standardising each column to mean 0 and standard deviation 1 — `scale()` in R — is what makes the sum meaningful. Even then, correlated columns count twice.

**Tanimoto (Jaccard) distance on binary fingerprints.** For bit vectors \(a,b\),
\[ S_{\text{Tanimoto}}(a,b)=\frac{|a \cap b|}{|a|+|b|-|a \cap b|}, \qquad d = 1 - S . \]
The numerator counts bits set in both molecules, the denominator bits set in either. Zeros shared by two molecules — both lack a nitro group, both lack a sugar — contribute nothing, which is the whole point: in a 2,048-bit fingerprint with about fifty bits set, shared absence is uninformative and Euclidean distance would be dominated by it<sup class="c"><a href="#r21">[21]</a></sup>. In sparse-matrix algebra the whole matrix is three lines: \(|a\cap b|\) for all pairs is the Gram matrix `tcrossprod(FP)`, \(|a|\) is `rowSums(FP)`, and the denominator is their outer sum minus the intersection.

**RMSD on conformations.** After optimal superposition, \(\text{RMSD}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}\|\mathbf{r}_i-\mathbf{r}'_i\|^2}\) over \(N\) selected atoms. It is a Euclidean distance in \(3N\) dimensions divided by \(\sqrt{N}\), which is why superposed coordinates can be handed straight to a Euclidean algorithm and the answer read in nanometres<sup class="c"><a href="#r22">[22]</a></sup>.

**Correlation distance on expression profiles.** \(d = 1-r\) with \(r\) the Pearson correlation between two profiles: two patients whose genes move together are close even if one array is globally brighter. Correlation distance discards the level and keeps the shape — usually right for microarray samples, never right for molecular weight.

### 2.1 A worked example by hand

Take two molecules whose folded fingerprints set the bit sets \(A=\{a,b,c,d\}\) and \(B=\{a,b\}\), so \(B\subset A\). Then \(|A\cap B| = 2\), \(|A\cup B| = 4\), and

\[ S = \frac{2}{4+2-2} = 0.5, \qquad d_{\text{Tanimoto}} = 0.5 . \]

The next chunk checks that arithmetic, then repeats it on the real 2,048-bit fingerprints of the first two ROCK1 compounds: the set operations are done explicitly with `intersect()` and `union()` and compared with the matrix-algebra implementation that the rest of the unit uses.



In [ ]:
A <- c(1, 2, 3, 4); B <- c(1, 2)
cat(sprintf("toy   |A|=%d |B|=%d |A and B|=%d |A or B|=%d  ->  S = %d/%d = %.4f, d = %.4f\n",
            length(A), length(B), length(intersect(A, B)), length(union(A, B)),
            length(intersect(A, B)), length(union(A, B)),
            length(intersect(A, B))/length(union(A, B)),
            1 - length(intersect(A, B))/length(union(A, B))))

tanimoto <- function(M) {                      # all pairwise Tanimoto similarities, sparse input
  inter <- as.matrix(tcrossprod(M)); n <- rowSums(M)
  inter / (outer(n, n, "+") - inter)
}
a1 <- which(FP[1, ] == 1); b1 <- which(FP[2, ] == 1)
i12 <- length(intersect(a1, b1)); u12 <- length(union(a1, b1))
cat(sprintf("ROCK1 rows 1 and 2: |A|=%d |B|=%d |A and B|=%d |A or B|=%d\n", length(a1), length(b1), i12, u12))
cat(sprintf("by hand  S = %.4f   d = %.4f\nmatrix   S = %.4f   d = %.4f\n",
            i12/u12, 1 - i12/u12, tanimoto(FP[1:2, ])[1, 2], 1 - tanimoto(FP[1:2, ])[1, 2]))
cat(sprintf("Euclidean distance between the same bit vectors %.2f; %d bits are zero in both and are ignored\n",
            sqrt(sum((FP[1, ] - FP[2, ])^2)), 2048 - u12))



The hand calculation and the matrix algebra agree exactly, which is the only way to be sure you know what a similarity function counts. Of 2,048 bits the overwhelming majority are zero in both molecules, so a Euclidean distance between the same vectors has no chemical scale, whereas the Tanimoto distance is bounded in \([0,1]\) and has a conventional reading — below about 0.35 (similarity above 0.65) two Morgan fingerprints usually belong to the same series<sup class="c"><a href="#r18">[18]</a></sup>.

### 2.2 The cost of a distance matrix

Most algorithms below need all \(n(n-1)/2\) pairwise distances: 3.8 million numbers for our 2,764 compounds (trivial), \(5\times10^9\) for a 100,000-compound catalogue (impossible on a laptop). Production pipelines therefore use sparse neighbour lists or algorithms that never materialise the matrix. Plan the memory before launching.



In [ ]:
rows <- list()
for (n in c(500, 1000, 2000, nrow(FP))) {
  t0 <- proc.time()[["elapsed"]]
  Dn <- 1 - tanimoto(FP[1:n, ])
  rows[[length(rows) + 1]] <- data.frame(
    n = n, pairs = n*(n - 1)/2, MB_double = round(n*(n - 1)/2*8/1e6, 1),
    seconds = round(proc.time()[["elapsed"]] - t0, 3))
}
knitr::kable(do.call(rbind, rows), caption = "Cost of the full Tanimoto distance matrix")
cat(sprintf("extrapolated to a 100,000-compound library: %.1f billion pairs = %.0f GB in double precision\n",
            1e5*99999/2/1e9, 1e5*99999/2*8/1e9))



The measured times confirm the quadratic law — quadrupling \(n\) multiplies the work by roughly sixteen — and the extrapolation to a vendor catalogue is the practical message: the algorithm must be chosen so that the matrix is never built. R stores doubles, not single-precision floats, so the memory bill is twice what the Python edition quotes for the same library.

## 3. k-means: the centroid family

k-means minimises the **within-cluster sum of squares** (inertia)

\[ J=\sum_{k=1}^{K}\sum_{i \in C_k} \| \mathbf{x}_i - \boldsymbol{\mu}_k \|^2 , \qquad \boldsymbol{\mu}_k=\frac{1}{|C_k|}\sum_{i \in C_k}\mathbf{x}_i . \]

Lloyd's algorithm<sup class="c"><a href="#r2">[2]</a></sup> alternates two steps that each can only decrease \(J\): **assign** every point to its nearest centroid, then **update** every centroid to the mean of its members. It converges to a local minimum that depends on the initial centroids, which is why `stats::kmeans()` is always called here with `nstart = 10` — ten random starts, best result kept. Cost per iteration is \(O(nKd)\) with no distance matrix, so k-means scales to millions of rows.

**Assumptions.** Minimising squared Euclidean distances to a mean implies clusters that are *spherical, of similar spread and of similar size*, and a mean that is a meaningful object: the mean of a set of fingerprints is not a molecule. Section 8 applies k-means to fingerprints deliberately, to show what it costs.

### 3.1 A worked example by hand

Six points on a line, \(x = 1, 2, 3, 10, 11, 12\), with \(K=2\) and a deliberately bad initialisation \(\mu_1 = 1\), \(\mu_2 = 2\).

| step | \(\mu_1\) | \(\mu_2\) | assignment | \(J\) |
|---|---|---|---|---|
| start | 1.00 | 2.00 | \(\{1\}\), \(\{2,3,10,11,12\}\) | \(0 + (2-7.6)^2+(3-7.6)^2+(10-7.6)^2+(11-7.6)^2+(12-7.6)^2 = 246.0\) |
| after update 1 | 1.00 | 7.60 | \(\{1,2,3\}\), \(\{10,11,12\}\) | \((1-1)^2 + \dots = 41.7\) |
| after update 2 | 2.00 | 11.00 | \(\{1,2,3\}\), \(\{10,11,12\}\) | \(2 + 2 = 4.0\) — converged |

The first row evaluates \(J\) with the *starting* centroids and the assignment they induce, which is why it is so large; the second row re-evaluates the same quantity after the centroids have moved. The chunk reproduces the trace and compares with `stats::kmeans`.



In [ ]:
x <- c(1, 2, 3, 10, 11, 12); mu <- c(1, 2)
for (step in 0:3) {
  lab <- apply(abs(outer(x, mu, "-")), 1, which.min)
  J   <- sum(sapply(1:2, function(k) sum((x[lab == k] - mu[k])^2)))
  cat(sprintf("step %d: centroids %-12s assignment %s  J = %6.1f\n", step,
              paste(round(mu, 2), collapse = " "), paste(lab, collapse = ""), J))
  new <- sapply(1:2, function(k) if (any(lab == k)) mean(x[lab == k]) else mu[k])
  if (isTRUE(all.equal(new, mu))) { cat("converged\n"); break }
  mu <- new
}
set.seed(0); km6 <- kmeans(matrix(x), centers = 2, nstart = 10)
cat(sprintf("\nstats::kmeans: centroids %s  within-cluster SS %.1f  sizes %s\n",
            paste(sort(round(km6$centers, 2)), collapse = " and "),
            km6$tot.withinss, paste(km6$size, collapse = "/")))



The hand trace and `stats::kmeans` agree: centroids 2 and 11, within-cluster sum of squares 4.0, two clusters of three. The intermediate line is the lesson — the partition is already correct after the first update, while \(J\) fell from 246.0 to 41.7 and only then to 4.0. A bad initialisation shows up as a stalled \(J\), not as an error message.

### 3.2 k-means on the compound descriptors

Now the real table: 2,764 compounds in 11 standardised descriptor dimensions, with \(K\) swept from 2 to 10 and five quantities recorded at each \(K\) — inertia, mean **silhouette**, **Davies-Bouldin**, **Calinski-Harabasz** and the size of the smallest cluster, the diagnostic most people forget. `cluster::silhouette()` needs a full distance object, so the silhouette is computed on a fixed random subsample of 1,200 compounds; Davies-Bouldin and Calinski-Harabasz are written out from their definitions, because base R ships neither.



In [ ]:
db_index <- function(X, lab) {                       # Davies-Bouldin, lower is better
  ks <- sort(unique(lab)); K <- length(ks)
  cen <- t(sapply(ks, function(k) colMeans(X[lab == k, , drop = FALSE])))
  S <- sapply(seq_along(ks), function(i) {
    Xi <- X[lab == ks[i], , drop = FALSE]
    mean(sqrt(rowSums(sweep(Xi, 2, cen[i, ])^2))) })
  M <- as.matrix(dist(cen)); diag(M) <- NA
  mean(sapply(1:K, function(i) max((S[i] + S)/M[i, ], na.rm = TRUE)))
}
ch_index <- function(X, lab) {                       # Calinski-Harabasz, higher is better
  ks <- unique(lab); K <- length(ks); n <- nrow(X); g <- colMeans(X)
  B <- sum(sapply(ks, function(k) sum(lab == k)*sum((colMeans(X[lab == k, , drop = FALSE]) - g)^2)))
  W <- sum(sapply(ks, function(k) { Xi <- X[lab == k, , drop = FALSE]
                                    sum(sweep(Xi, 2, colMeans(Xi))^2) }))
  (B/(K - 1)) / (W/(n - K))
}
ari <- function(a, b) {                              # adjusted Rand index, Hubert and Arabie
  tab <- table(a, b); ch2 <- function(m) sum(m*(m - 1)/2)
  idx <- ch2(tab); ea <- ch2(rowSums(tab)); eb <- ch2(colSums(tab)); e <- ea*eb/ch2(sum(tab))
  (idx - e)/((ea + eb)/2 - e)
}
cat("three validity functions defined:", paste(c("db_index", "ch_index", "ari"), collapse = ", "), "\n")



The three functions are short enough to read, and reading them is the point: Davies-Bouldin<sup class="c"><a href="#r4">[4]</a></sup> averages over clusters the worst ratio of within-cluster scatter to between-centroid distance, Calinski-Harabasz<sup class="c"><a href="#r5">[5]</a></sup> is a between/within variance ratio corrected for degrees of freedom, and the adjusted Rand index<sup class="c"><a href="#r15">[15]</a></sup> counts object *pairs* treated consistently by two partitions, minus the agreement expected by chance (verified numerically in Section 7.2).



In [ ]:
set.seed(0); sil_sub <- sample(nrow(Zdesc), 1200); Dsil <- dist(Zdesc[sil_sub, ])
sweep_rows <- list()
for (K in 2:10) {
  set.seed(0); km <- kmeans(Zdesc, centers = K, nstart = 10, iter.max = 50)
  sweep_rows[[length(sweep_rows) + 1]] <- data.frame(
    K = K, inertia = km$tot.withinss,
    silhouette = mean(silhouette(km$cluster[sil_sub], Dsil)[, 3]),
    davies_bouldin = db_index(Zdesc, km$cluster),
    calinski_harabasz = ch_index(Zdesc, km$cluster),
    smallest_cluster = min(km$size))
}
swp <- do.call(rbind, sweep_rows)
knitr::kable(swp, digits = 3, caption = "k-means on 11 standardised descriptors")
best_sil <- swp$K[which.max(swp$silhouette)]; best_db <- swp$K[which.min(swp$davies_bouldin)]
cat(sprintf("silhouette prefers K = %d; Davies-Bouldin prefers K = %d; inertia prefers the largest K on offer, as it always does\n",
            best_sil, best_db))
cat(sprintf("silhouette range over the sweep: %.3f to %.3f; smallest cluster at K = 10: %d compounds\n",
            min(swp$silhouette), max(swp$silhouette), swp$smallest_cluster[swp$K == 10]))


In [ ]:
g1 <- ggplot(swp, aes(K, inertia)) + geom_line(colour = PAL["blue"]) +
  geom_point(colour = PAL["blue"]) + labs(y = "inertia (within-cluster SS)", title = "Inertia: monotone")
g2 <- ggplot(swp, aes(K, silhouette)) + geom_line(colour = PAL["orange"]) +
  geom_point(colour = PAL["orange"]) + geom_vline(xintercept = best_sil, linetype = 2, colour = "grey40") +
  labs(y = "mean silhouette", title = sprintf("Silhouette: max at K = %d", best_sil))
g3 <- ggplot(swp, aes(K, davies_bouldin)) + geom_line(colour = PAL["green"]) +
  geom_point(colour = PAL["green"]) + geom_vline(xintercept = best_db, linetype = 2, colour = "grey40") +
  labs(y = "Davies-Bouldin (lower better)", title = sprintf("Davies-Bouldin: min at K = %d", best_db))
print(g1 | g2 | g3)



<p class="caption"><b>Figure 1.</b> k-means on 2,764 compounds in 11 standardised descriptor dimensions. Inertia (left) falls monotonically from about 20,500 at K = 2 and cannot select K; the mean silhouette (centre) peaks at the smallest K on offer, where Davies-Bouldin (right) also has its minimum. The silhouette stays below 0.3: descriptor space is a continuum, not separated groups.</p>

The silhouette never rises above about 0.29 and falls to 0.19: the average compound is almost as close to its neighbouring cluster as to its own, so the partition is a convenient subdivision of a cloud, not a discovery of structure. Note also the last column — from \(K=8\) onwards k-means is spending centroids on clusters of a dozen compounds. Reporting "we identified K chemical classes by k-means" from such a sweep is the commonest unsupervised-learning error in the medicinal-chemistry literature.

### 3.3 Three geometries where k-means fails

The assumptions are easy to state and easy to violate. The next chunk builds four synthetic data sets where the ground truth is known by construction, and compares k-means with the truth by the adjusted Rand index: 1.0 is a perfect match, 0.0 is chance.



In [ ]:
make_blobs <- function(n, centers, sd, seed = 0) {
  set.seed(seed); K <- nrow(centers); lab <- rep(1:K, length.out = n)
  sdv <- if (length(sd) == 1) rep(sd, K) else sd
  list(X = t(sapply(seq_len(n), function(i) rnorm(2, centers[lab[i], ], sdv[lab[i]]))), y = lab)
}
b  <- make_blobs(600, matrix(c(0,0, 2.8,2.4, 5.2,-0.6), ncol = 2, byrow = TRUE), 0.8, seed = 1)
Xa <- b$X %*% matrix(c(0.61, -0.41, -0.64, 0.85), 2, 2)                      # sheared
bv <- make_blobs(600, matrix(c(0,0, 3.5,3.0, 7,0), ncol = 2, byrow = TRUE), c(1, 3, 0.4), seed = 2)
bs <- make_blobs(1500, matrix(c(0,0, 2.2,2.2, -2.2,2.2), ncol = 2, byrow = TRUE), 0.9, seed = 3)
ks <- c(which(bs$y == 1)[1:500], which(bs$y == 2)[1:50], which(bs$y == 3)[1:50])
set.seed(5); tt <- runif(300, 0, pi)
Xm <- rbind(cbind(cos(tt), sin(tt)), cbind(1 - cos(tt), 0.5 - sin(tt))) +
      matrix(rnorm(1200, 0, 0.06), ncol = 2)
cases <- list("anisotropic" = list(Xa, b$y), "unequal variance" = list(bv$X, bv$y),
              "unequal size 500/50/50" = list(bs$X[ks, ], bs$y[ks]), "two moons" = list(Xm, rep(1:2, each = 300)))
for (nm in names(cases)) {
  Xc <- cases[[nm]][[1]]; yc <- cases[[nm]][[2]]
  set.seed(0); lab <- kmeans(Xc, length(unique(yc)), nstart = 10)$cluster
  cat(sprintf("%-24s K=%d  ARI(k-means, truth) = %.3f  cluster sizes %s\n", nm, length(unique(yc)),
              ari(yc, lab), paste(sort(table(lab), decreasing = TRUE), collapse = "/")))
}


In [ ]:
plots <- list()
for (nm in names(cases)) {
  Xc <- cases[[nm]][[1]]; yc <- cases[[nm]][[2]]
  set.seed(0); lab <- kmeans(Xc, length(unique(yc)), nstart = 10)$cluster
  d1 <- data.frame(x = Xc[, 1], y = Xc[, 2], g = factor(yc))
  d2 <- data.frame(x = Xc[, 1], y = Xc[, 2], g = factor(lab))
  base <- function(d, ti) ggplot(d, aes(x, y, colour = g)) + geom_point(size = .5, show.legend = FALSE) +
    scale_colour_manual(values = unname(PAL)) + labs(title = ti, x = NULL, y = NULL) +
    theme(axis.text = element_blank(), plot.title = element_text(size = 8))
  plots[[length(plots) + 1]] <- base(d1, paste0(nm, "\ntruth"))
  plots[[length(plots) + 1]] <- base(d2, sprintf("k-means, ARI = %.2f", ari(yc, lab)))
}
print(wrap_plots(plots[c(1,3,5,7,2,4,6,8)], nrow = 2))



<p class="caption"><b>Figure 2.</b> Top row, ground truth; bottom row, k-means with the correct K. Sheared clusters are cut across their long axis (ARI 0.48); a wide cluster beside a narrow one is dismembered (0.58); a 500/50/50 split is rebalanced towards equal sizes, the printed sizes becoming 354/155/91 (ARI 0.32); two interleaved crescents defeat any straight-boundary partition (0.21).</p>

Each failure has a fix: shear and unequal variance are handled by a Gaussian mixture with a richer covariance (Section 6), unequal sizes by a density method that does not try to balance, non-convex shapes only by density or spectral methods. No amount of tuning \(K\) repairs any of them, and the printed cluster sizes are the diagnostic: when the recovered sizes are more equal than the truth, suspect the objective rather than the data.

<div class="exercise"><h4>Exercise 1</h4>
<p>Re-run the anisotropic case after whitening the data (<code>prcomp(Xa, scale. = TRUE)$x</code>, or divide each principal component by its standard deviation) and recompute the ARI against the truth. Whitening removes the global shear but not a per-cluster one: explain, from the printed cluster sizes, why the recovery is only partial. Then increase the <code>sd</code> argument of <code>make_blobs</code> until k-means recovers nothing (ARI below 0.1) and report the value.</p></div>

## 4. Hierarchical clustering: the whole family of partitions at once

Agglomerative clustering starts with \(n\) singletons and repeatedly merges the two closest, producing a binary tree (**dendrogram**) whose every horizontal cut is a partition: you choose \(K\) afterwards, a real advantage when \(K\) is the question. "Closest" needs a definition for *sets*, and that is the **linkage**:

| linkage | distance between clusters \(A,B\) | behaviour |
|---|---|---|
| single | \(\min_{a\in A,b\in B} d(a,b)\) | follows filaments; chains clusters together through bridges of points |
| complete | \(\max_{a\in A,b\in B} d(a,b)\) | compact, equal-diameter clusters; sensitive to outliers |
| average (UPGMA) | mean of all pairs | compromise; the chemoinformatics default |
| Ward | increase in within-cluster sum of squares caused by the merge<sup class="c"><a href="#r7">[7]</a></sup> | spherical, similar-sized clusters; only valid with Euclidean distances<sup class="c"><a href="#r8">[8]</a></sup> |

Cost is \(O(n^2)\) memory, so a few tens of thousands of objects is the ceiling. R exposes all four through `hclust()`; note the R-specific trap that `method = "ward.D2"`, not `"ward.D"`, implements Ward's criterion on a distance (rather than squared-distance) input — the distinction Murtagh and Legendre analyse<sup class="c"><a href="#r8">[8]</a></sup>. The **cophenetic correlation** correlates each pair's original distance with the height at which the pair is first merged: near 1 the dendrogram is a faithful summary, low values mean the tree is an artefact of the linkage rule.



In [ ]:
set.seed(0); sub <- sort(sample(nrow(Zdesc), 600)); Dsub <- dist(Zdesc[sub, ])
trees <- list(); hrows <- list()
for (meth in c("single", "complete", "average", "ward.D2")) {
  Lk <- hclust(Dsub, method = meth); trees[[meth]] <- Lk
  l4 <- cutree(Lk, 4)
  hrows[[length(hrows) + 1]] <- data.frame(
    linkage = meth, cophenetic_r = round(cor(Dsub, cophenetic(Lk)), 3),
    sizes_at_K4 = paste(sort(table(l4), decreasing = TRUE), collapse = "/"),
    singletons_at_K4 = sum(table(l4) == 1))
}
knitr::kable(do.call(rbind, hrows), row.names = FALSE,
             caption = "Four linkages on the same 600 compounds")

op <- par(mfrow = c(1, 4), mar = c(1, 4, 3, 1))
for (meth in names(trees)) {
  plot(trees[[meth]], labels = FALSE, hang = -1, xlab = "", sub = "", ylab = "merge height",
       main = sprintf("%s\ncophenetic r = %.2f", meth, cor(Dsub, cophenetic(trees[[meth]]))),
       cex.main = 0.95)
}
par(op)



<p class="caption"><b>Figure 3.</b> Dendrograms of the same 600 compounds under four linkages. Single linkage (left) shows the staircase of chaining — almost every merge adds one more point to one growing cluster, so a cut at K = 4 gives one cluster of 596 plus two singletons. Complete and average linkage give trees with a longer trunk, Ward the most balanced of all (183/147/138/132). The cophenetic correlations differ by linkage on identical data, from 0.44 for Ward to 0.77 for average linkage.</p>

The table makes the chaining concrete: at \(K=4\) single and average linkage return one giant cluster beside singletons, while Ward returns four groups of comparable size. Neither is "right" — single linkage suits filaments (a congeneric series extended one substituent at a time really is a chain), Ward suits clusters you intend to summarise by a centroid. Note the inversion worth remembering: average linkage has the *highest* cophenetic correlation (0.774) and the *least* usable four-cluster partition, because reproducing the distances of a continuum and cutting that continuum into useful pieces are different achievements. Choosing the linkage after seeing which gives the prettiest picture is a researcher degree of freedom: decide before you look.

## 5. Density-based clustering: DBSCAN and OPTICS, written by hand

Density methods define a cluster as a connected region of high density and label the rest **noise**. DBSCAN<sup class="c"><a href="#r9">[9]</a></sup> has two parameters, a radius \(\varepsilon\) and a count `minPts`: a point is a *core point* if at least `minPts` points lie within \(\varepsilon\); core points within \(\varepsilon\) of each other join; non-core points within \(\varepsilon\) of a core point are border points; everything else is noise.

Three properties follow: the number of clusters is an output, not a parameter; clusters may have any shape, because connectivity replaces convexity; and a point may be assigned to no cluster at all — what you want for screening data full of singleton chemotypes, and what breaks code that assumes every row has a label.

<div class="box bridge"><h4>Substitution — no <code>dbscan</code> package here</h4>
<p>The R package <code>dbscan</code> (which provides DBSCAN, OPTICS and HDBSCAN) is <strong>not installed</strong> in this course environment, so both algorithms are written out below in about fifteen lines each. That is a gain rather than a loss for a teaching unit — the whole of DBSCAN is a neighbour list, a core-point test and a breadth-first search — but two practical consequences must be stated. First, the hand-written versions are \(O(n^2)\): they build the full distance matrix, which is fine for the 2,764 compounds here and hopeless for a catalogue, whereas the package uses a k-d tree. Second, <strong>HDBSCAN is not reproduced</strong>: its value is the variable-density cluster extraction from a condensed tree<sup class="c"><a href="#r11">[11]</a></sup>, which is far more than fifteen lines, so Section 5.2 demonstrates the problem HDBSCAN solves rather than the solution.</p></div>



In [ ]:
dbscan_hand <- function(D, eps, minPts) {            # returns 0 for noise, 1..k for clusters
  n <- nrow(D); nb <- lapply(seq_len(n), function(i) which(D[i, ] <= eps))
  core <- lengths(nb) >= minPts; lab <- integer(n); cid <- 0L
  for (i in seq_len(n)) {
    if (lab[i] != 0L || !core[i]) next
    cid <- cid + 1L; lab[i] <- cid; q <- nb[[i]]
    while (length(q)) {
      j <- q[1]; q <- q[-1]
      if (lab[j] == 0L) { lab[j] <- cid; if (core[j]) q <- c(q, setdiff(nb[[j]], q)) }
    }
  }
  lab
}
MINPTS <- 2*ncol(Zdesc)                              # the usual rule of thumb: 2 x dimensionality
Dz <- as.matrix(dist(Zdesc))
kd <- sort(apply(Dz, 1, function(r) sort(r)[MINPTS + 1]))
cat(sprintf("minPts = %d; %d-th neighbour distance: median %.2f, 95th percentile %.2f, max %.2f\n",
            MINPTS, MINPTS, median(kd), quantile(kd, 0.95), max(kd)))
dbrows <- do.call(rbind, lapply(c(0.8, 1.0, 1.2, 1.5, 2.0, 2.5), function(eps) {
  lab <- dbscan_hand(Dz, eps, MINPTS)
  data.frame(eps = eps, clusters = max(lab), noise_pct = round(100*mean(lab == 0), 1),
             largest_pct = round(100*max(c(0, table(lab[lab > 0])))/length(lab), 1)) }))
knitr::kable(dbrows, row.names = FALSE, caption = "DBSCAN on the descriptor space, eps swept")



The weakness is structural: a single global \(\varepsilon\) cannot serve regions of different density. The standard heuristic is the **k-distance plot** — sort every point's distance to its `minPts`-th nearest neighbour and look for a knee<sup class="c"><a href="#r9">[9]</a></sup>. OPTICS<sup class="c"><a href="#r10">[10]</a></sup> avoids committing to one \(\varepsilon\) by ordering the points so that the *reachability* profile shows clusters as valleys; it is implemented below with a simple seed list, which is all the algorithm is.



In [ ]:
optics_hand <- function(D, minPts) {
  n <- nrow(D); cd <- apply(D, 1, function(r) sort(r)[minPts + 1L])   # core distances
  done <- logical(n); reach <- rep(Inf, n); ord <- integer(0); seeds <- integer(0)
  update <- function(q) {
    cand <- which(!done); newr <- pmax(cd[q], D[q, cand]); better <- newr < reach[cand]
    reach[cand[better]] <<- newr[better]; seeds <<- union(seeds, cand[better])
  }
  for (p in seq_len(n)) {
    if (done[p]) next
    done[p] <- TRUE; ord <- c(ord, p); seeds <- integer(0); update(p)
    while (length(seeds)) {
      q <- seeds[which.min(reach[seeds])]; seeds <- setdiff(seeds, q)
      done[q] <- TRUE; ord <- c(ord, q); update(q)
    }
  }
  list(order = ord, reach = reach[ord], coredist = cd)
}
op_res <- optics_hand(Dz, MINPTS)
cat(sprintf("OPTICS ordering of %d compounds; reachability: median %.2f, 90th pct %.2f, max %.2f\n",
            length(op_res$order), median(op_res$reach[is.finite(op_res$reach)]),
            quantile(op_res$reach[is.finite(op_res$reach)], .9), max(op_res$reach[is.finite(op_res$reach)])))


In [ ]:
g1 <- ggplot(data.frame(i = seq_along(kd), d = kd), aes(i, d)) + geom_line(colour = PAL["blue"]) +
  geom_hline(yintercept = quantile(kd, .95), linetype = 2, colour = PAL["orange"]) +
  labs(x = "points sorted by distance", y = sprintf("distance to %d-th neighbour", MINPTS),
       title = "k-distance plot: no sharp knee")
g2 <- ggplot(dbrows, aes(eps)) +
  geom_line(aes(y = clusters), colour = PAL["blue"]) + geom_point(aes(y = clusters), colour = PAL["blue"]) +
  geom_line(aes(y = noise_pct/10), colour = PAL["orange"], linetype = 2) +
  geom_point(aes(y = noise_pct/10), colour = PAL["orange"], shape = 15) +
  scale_y_continuous(name = "number of clusters",
                     sec.axis = sec_axis(~ .*10, name = "% noise (dashed)")) +
  labs(title = "DBSCAN is a trade-off, not a discovery")
g3 <- ggplot(data.frame(i = seq_along(op_res$reach), r = pmin(op_res$reach, 6)), aes(i, r)) +
  geom_line(linewidth = .2, colour = "grey30") +
  labs(x = "OPTICS ordering", y = "reachability distance", title = "OPTICS: valleys are clusters")
print(g1 | g2 | g3)



<p class="caption"><b>Figure 4.</b> Left: the k-distance curve rises smoothly with no sharp knee — this cloud has no characteristic density scale. Centre: as eps grows from 0.8 to 2.5 the cluster count falls from 7 to 1 while the noise fraction falls from 83 % to 2 %, a pure trade-off with no plateau. Right: the OPTICS reachability profile (clipped at 6) is similarly featureless, with a median reachability of about 1.1 and no deep, wide valleys.</p>

That negative result is worth more than a forced positive one: with no knee, no plateau and no valleys, the correct sentence is "no density-separated groups were found in descriptor space", not "DBSCAN identified five clusters (eps = 1.2)". To see the same algorithm succeed — and then fail for a different reason — apply it to two geometries where the answer is known.

### 5.1 Where density methods shine, and where a single eps is not enough



In [ ]:
set.seed(7)
Xd <- rbind(matrix(rnorm(600, 0, 0.25), ncol = 2),                      # tight blob
            cbind(rnorm(300, 4, 1.2), rnorm(300, 4, 1.2)),              # loose blob
            cbind(runif(120, -6, 10), runif(120, -6, 10)))              # uniform background
yd <- c(rep(1, 300), rep(2, 300), rep(0, 120))
Dm <- as.matrix(dist(Xm)); Dd <- as.matrix(dist(Xd))
res <- list()
for (nm in c("two moons", "two densities + noise")) {
  Dc <- if (nm == "two moons") Dm else Dd
  Xc <- if (nm == "two moons") Xm else Xd
  yc <- if (nm == "two moons") rep(1:2, each = 300) else yd
  set.seed(0); res[[paste(nm, "k-means")]] <- kmeans(Xc, 2, nstart = 10)$cluster
  for (eps in if (nm == "two moons") c(0.3) else c(0.2, 0.6, 1.0))
    res[[sprintf("%s DBSCAN eps=%.1f", nm, eps)]] <- dbscan_hand(Dc, eps, 5)
}
for (nm in names(res)) {
  lab <- res[[nm]]
  yc <- if (startsWith(nm, "two moons")) rep(1:2, each = 300) else yd
  cat(sprintf("%-34s clusters %2d  noise %5.1f%%  ARI vs truth %.3f\n",
              nm, length(setdiff(unique(lab), 0)), 100*mean(lab == 0), ari(yc, lab)))
}


In [ ]:
show <- c("two moons k-means", "two moons DBSCAN eps=0.3", "two densities + noise k-means",
          "two densities + noise DBSCAN eps=0.2", "two densities + noise DBSCAN eps=0.6",
          "two densities + noise DBSCAN eps=1.0")
pl <- lapply(show, function(nm) {
  Xc <- if (startsWith(nm, "two moons")) Xm else Xd
  lab <- res[[nm]]
  d <- data.frame(x = Xc[, 1], y = Xc[, 2], g = factor(lab), noise = lab == 0)
  ggplot(d, aes(x, y)) +
    geom_point(data = subset(d, noise), colour = "grey75", shape = 4, size = .6) +
    geom_point(data = subset(d, !noise), aes(colour = g), size = .6, show.legend = FALSE) +
    labs(title = sprintf("%s\n%d clusters, %.0f%% noise", nm, length(setdiff(unique(lab), 0)),
                         100*mean(lab == 0)), x = NULL, y = NULL) +
    theme(axis.text = element_blank(), plot.title = element_text(size = 7.5))
})
print(wrap_plots(pl, nrow = 2))



<p class="caption"><b>Figure 5.</b> Grey crosses are points labelled noise. Top: on the two crescents k-means cuts straight through both (ARI 0.21) while the hand-written DBSCAN recovers them exactly (ARI 1.000). Bottom: a tight blob, a loose blob and a uniform background, under k-means and three values of eps — 0.2 shatters the loose blob and calls 40 % of the data noise (ARI 0.56), 0.6 recovers both blobs (ARI 0.91), and 1.0 bridges the background into one 622-point cluster (ARI 0.27).</p>

The bottom row makes the structural weakness of a single global radius visible: the window of \(\varepsilon\) that serves *both* densities is narrow, and nothing in the data tells you where it is. It exists at all only because the two blobs differ in spread by a factor of about five. That is the problem HDBSCAN<sup class="c"><a href="#r11">[11]</a></sup> solves by building the hierarchy over all density levels and keeping the clusters that persist longest — the reason the `dbscan` package is worth installing in a real project.

## 6. Model-based clustering: a Gaussian mixture by hand, and BIC

A Gaussian mixture model (GMM) assumes the data were generated by \(K\) multivariate normals,

\[ p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k), \qquad \sum_k \pi_k = 1,\]

and fits them by **expectation-maximisation**<sup class="c"><a href="#r12">[12]</a></sup>: the E-step computes each point's *responsibility* \(\gamma_{ik}\) (the posterior probability that component \(k\) generated point \(i\)), the M-step re-estimates the parameters as responsibility-weighted statistics. k-means is the limit \(\boldsymbol{\Sigma}_k = \sigma^2 I\), \(\sigma \to 0\). Being a probability model, a GMM has a likelihood and hence a principled way to compare \(K\) *and* covariance structures: the **Bayesian information criterion**<sup class="c"><a href="#r14">[14]</a></sup>

\[\mathrm{BIC} = -2\ln \hat{L} + p \ln n,\]

where \(p\) is the number of free parameters. Model-based clustering with BIC selection<sup class="c"><a href="#r13">[13]</a></sup> is the closest thing clustering has to supervised model selection — with the caveat that it answers "how many Gaussians describe this density best", which is the same question as "how many groups are there" only if the groups are Gaussian. The package `mclust`, which fits the full family of fourteen covariance parameterisations, is **not installed here**, so the EM is written out below for the two structures that are a few lines each: spherical (\(\sigma^2_k I\)) and diagonal.



In [ ]:
gmm_em <- function(X, K, type = c("diag", "spherical"), iters = 120, seed = 0, tol = 1e-7) {
  type <- match.arg(type); set.seed(seed); n <- nrow(X); d <- ncol(X)
  km <- kmeans(X, K, nstart = 5, iter.max = 50)                      # k-means start
  mu <- matrix(km$centers, K, d); va <- matrix(rep(apply(X, 2, var), each = K), K, d)
  if (type == "spherical") va[] <- rowMeans(va)
  pi_ <- as.numeric(table(km$cluster))/n; ll_old <- -Inf
  for (it in seq_len(iters)) {
    logp <- matrix(sapply(1:K, function(k) log(pi_[k]) -
      0.5*rowSums(sweep(sweep(X, 2, mu[k, ])^2, 2, va[k, ], "/")) -
      0.5*sum(log(2*pi*va[k, ]))), n, K)
    m <- apply(logp, 1, max); ll <- sum(m + log(rowSums(exp(logp - m))))   # E-step
    g <- exp(logp - m); g <- g/rowSums(g); Nk <- colSums(g) + 1e-10
    pi_ <- Nk/n                                                            # M-step
    mu <- matrix(t(sapply(1:K, function(k) colSums(g[, k]*X)/Nk[k])), K, d)
    va <- matrix(t(sapply(1:K, function(k) colSums(g[, k]*sweep(X, 2, mu[k, ])^2)/Nk[k])), K, d) + 1e-6
    if (type == "spherical") va[] <- rowMeans(va)
    if (abs(ll - ll_old) < tol*abs(ll)) break
    ll_old <- ll
  }
  np <- if (type == "spherical") K*d + K + (K - 1) else 2*K*d + (K - 1)
  list(loglik = ll, bic = -2*ll + np*log(n), resp = g, npar = np, iters = it)
}
fit1 <- gmm_em(Zdesc, 3, "diag")
cat(sprintf("diagonal GMM, K = 3: converged in %d EM iterations, log-likelihood %.0f, %d parameters, BIC %.0f\n",
            fit1$iters, fit1$loglik, fit1$npar, fit1$bic))


In [ ]:
bic_rows <- list()
for (ty in c("spherical", "diag")) for (K in 1:8) {
  f <- gmm_em(Zdesc, K, ty)
  bic_rows[[length(bic_rows) + 1]] <- data.frame(type = ty, K = K, bic = f$bic, npar = f$npar)
}
BT <- do.call(rbind, bic_rows)
knitr::kable(round(matrix(BT$bic/1000, 2, 8, byrow = TRUE,
                          dimnames = list(c("spherical", "diagonal"), paste0("K=", 1:8))), 1),
             caption = "BIC in thousands (lower is better)")
bst <- BT[which.min(BT$bic), ]
fitb <- gmm_em(Zdesc, bst$K, as.character(bst$type)); mx <- apply(fitb$resp, 1, max)
cat(sprintf("lowest BIC: '%s' covariance with K = %d (BIC = %.0f, %d parameters)\n",
            bst$type, bst$K, bst$bic, bst$npar))
cat(sprintf("mean maximum responsibility = %.3f; compounds whose best component has responsibility < 0.8: %.1f%%\n",
            mean(mx), 100*mean(mx < 0.8)))

g1 <- ggplot(BT, aes(K, bic/1000, colour = type, shape = type)) + geom_line() + geom_point() +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(y = "BIC (thousands, lower better)", title = "BIC selects K and the covariance structure") +
  theme(legend.position = "bottom", legend.title = element_blank())
g2 <- ggplot(data.frame(r = mx), aes(r)) +
  geom_histogram(bins = 40, fill = PAL["blue"], colour = NA) +
  geom_vline(xintercept = 0.8, linetype = 2, colour = PAL["orange"]) +
  labs(x = "maximum responsibility of the best component", y = "compounds",
       title = "Soft assignments: how sure is the model?")
print(g1 | g2)



<p class="caption"><b>Figure 6.</b> Left: BIC against K for the two covariance structures the hand-written EM fits; the diagonal family is far better than the spherical one at every K beyond 1, and its curve keeps falling to the largest K examined (36,100 at K = 8 against 86,400 at K = 1). Right: each compound's maximum responsibility, with the 0.8 line marking the compounds the model cannot assign confidently; the distribution is pressed against 1.0.</p>

The two diagnostics together tell the same story as Section 3: a flexible density model fits better and better as you give it more components. **A BIC that keeps falling is not evidence of many clusters; it is evidence that the density is not a mixture of a few well-separated Gaussians.** The responsibility histogram adds an R-specific caution to the Python edition's reading: a *diagonal* mixture with eight narrow components assigns almost every compound confidently (only 3.3 % below 0.8), not because the groups are real but because narrow components tile a continuum tightly. The Python edition, which fits full covariance matrices with `scikit-learn`, reports a much larger ambiguous fraction; the useful by-product is the same in both — the soft assignment tells you *which* compounds sit between components, and those are exactly the ones you should not describe as members of a series.

<div class="exercise"><h4>Exercise 2</h4>
<p>Fit the diagonal mixture with K = 3 to the anisotropic synthetic data of Section 3.3 and compute the ARI against the truth with <code>apply(fit$resp, 1, which.max)</code>. Compare it with the k-means ARI of 0.48 printed there. Then explain why a <em>diagonal</em> mixture cannot fully repair a sheared cluster, and what covariance structure would (one word), and confirm your reasoning by rotating the data 45 degrees before fitting.</p></div>

## 7. Validity: is this partition real?

Four questions, four tools.

**Internal — compact and separated?** The **silhouette** of point \(i\)<sup class="c"><a href="#r3">[3]</a></sup> is

\[ s(i) = \frac{b(i)-a(i)}{\max\{a(i),b(i)\}} \in [-1,1], \]

with \(a(i)\) the mean distance from \(i\) to its own cluster and \(b(i)\) to the nearest *other* cluster: \(s \approx 1\) means comfortably inside, \(s \approx 0\) on the boundary, \(s<0\) in the wrong cluster. Davies-Bouldin and Calinski-Harabasz were defined in Section 3.2. All three presuppose the kind of cluster k-means makes and penalise elongated DBSCAN clusters, so never use them to compare a density method with a centroid method.

**The elbow, and why it is not evidence.** The inertia curve always falls, so the eye is invited to find the \(K\) where it stops falling *fast*. The trouble is that smooth, structureless data produces the same bend. The **gap statistic**<sup class="c"><a href="#r6">[6]</a></sup> turns the heuristic into a comparison by measuring how much lower the observed \(\log\) inertia is than the average \(\log\) inertia of uniform reference data with the same bounding box,

\[ \mathrm{Gap}(K)=\mathbb{E}^{*}[\log W_K] - \log W_K ,\]

and choosing the smallest \(K\) whose gap is within one standard error of the next. Run it on data that truly has no clusters and the gap stays flat at zero, which is the control every elbow plot is missing.



In [ ]:
inertia_curve <- function(X, kmax = 8) sapply(1:kmax, function(k) {
  set.seed(0); kmeans(X, k, nstart = 5, iter.max = 50)$tot.withinss })
gap_stat <- function(X, kmax = 8, B = 5, seed = 0) {
  set.seed(seed); lo <- apply(X, 2, min); hi <- apply(X, 2, max)
  t(sapply(1:kmax, function(k) {
    set.seed(0); wk <- log(kmeans(X, k, nstart = 5, iter.max = 50)$tot.withinss)
    refs <- sapply(1:B, function(b) {
      U <- sapply(seq_len(ncol(X)), function(j) runif(nrow(X), lo[j], hi[j]))
      set.seed(0); log(kmeans(U, k, nstart = 5, iter.max = 50)$tot.withinss) })
    c(mean(refs) - wk, sd(refs)*sqrt(1 + 1/B)) }))
}
set.seed(11)
U <- sapply(seq_len(ncol(Zdesc)), function(j) runif(nrow(Zdesc), min(Zdesc[, j]), max(Zdesc[, j])))
Ireal <- inertia_curve(Zdesc); Iunif <- inertia_curve(U)
Greal <- gap_stat(Zdesc);      Gunif <- gap_stat(U, seed = 1)
gapdf <- data.frame(K = 1:8, inertia_real = Ireal/Ireal[1], inertia_uniform = Iunif/Iunif[1],
                    gap_real = Greal[, 1], gap_uniform = Gunif[, 1], se_uniform = Gunif[, 2])
knitr::kable(gapdf, digits = 3, row.names = FALSE,
             caption = "Normalised inertia and the gap statistic, real data and uniform noise")


In [ ]:
g1 <- ggplot(gapdf, aes(K)) +
  geom_line(aes(y = inertia_real, colour = "ROCK1 descriptors")) +
  geom_point(aes(y = inertia_real, colour = "ROCK1 descriptors")) +
  geom_line(aes(y = inertia_uniform, colour = "uniform noise, same box"), linetype = 2) +
  geom_point(aes(y = inertia_uniform, colour = "uniform noise, same box"), shape = 15) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(y = "inertia / inertia(K = 1)", title = "Both curves have an 'elbow'") +
  theme(legend.position = "bottom", legend.title = element_blank())
g2 <- ggplot(gapdf, aes(K)) +
  geom_line(aes(y = gap_real, colour = "ROCK1 descriptors")) +
  geom_point(aes(y = gap_real, colour = "ROCK1 descriptors")) +
  geom_errorbar(aes(ymin = gap_uniform - se_uniform, ymax = gap_uniform + se_uniform,
                    colour = "uniform noise"), width = .2) +
  geom_point(aes(y = gap_uniform, colour = "uniform noise"), shape = 15) +
  geom_hline(yintercept = 0, linewidth = .3) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(y = "gap statistic", title = "Gap: zero means 'no more structure than noise'") +
  theme(legend.position = "bottom", legend.title = element_blank())
print(g1 | g2)



<p class="caption"><b>Figure 7.</b> Left: inertia normalised by its K = 1 value for the real descriptors (blue) and for uniform noise in the same bounding box (orange); both bend in the same place, so the shape of an elbow is not evidence of clusters. Right: the gap statistic sits within about one standard error of zero at every K for the uniform reference — the correct answer — while for the real data it rises steadily and singles out no K.</p>

Read the two panels together: the elbow heuristic would have reported "three descriptor classes" for data generated by `runif`, while the gap statistic, carrying its own null model, refuses to. Every clustering claim needs one such control.

### 7.1 Silhouette plots: look at the distribution, not the mean

A mean silhouette hides everything. The per-point silhouette shows how many points are actually negative — members assigned to a cluster they do not belong to.



In [ ]:
sil_rows <- list(); sil_long <- list()
for (K in 2:5) {
  set.seed(0); lab <- kmeans(Zdesc, K, nstart = 10, iter.max = 50)$cluster
  sv <- silhouette(lab[sil_sub], Dsil)[, 3]
  sil_rows[[length(sil_rows) + 1]] <- data.frame(
    K = K, mean = mean(sv), median = median(sv), negative_pct = 100*mean(sv < 0),
    worst_cluster_mean = min(tapply(sv, lab[sil_sub], mean)))
  ordv <- order(lab[sil_sub], sv)
  sil_long[[length(sil_long) + 1]] <- data.frame(
    K = sprintf("K = %d (mean %.3f)", K, mean(sv)), i = seq_along(sv),
    s = sv[ordv], cl = factor(lab[sil_sub][ordv]))
}
knitr::kable(do.call(rbind, sil_rows), digits = 3, row.names = FALSE,
             caption = "Silhouette distribution on 1,200 compounds")

sl <- do.call(rbind, sil_long)
print(ggplot(sl, aes(i, s, fill = cl)) + geom_col(width = 1, show.legend = FALSE) +
        facet_wrap(~ K, nrow = 1, scales = "free_x") +
        scale_fill_manual(values = rep(unname(PAL), 2)) +
        geom_hline(yintercept = 0, linewidth = .3) + coord_flip() +
        labs(x = "compounds, sorted within cluster", y = "silhouette") +
        theme(axis.text.y = element_blank(), strip.text = element_text(size = 8)))



<p class="caption"><b>Figure 8.</b> Per-compound silhouettes, sorted within each cluster, for K = 2 to 5 on the 1,200-compound subsample. A healthy clustering shows wide knives of comparable length to the right of zero; here between 10 % and 21 % of compounds are negative at every K and at least one cluster has a mean near zero (0.05 at K = 5).</p>

### 7.2 External validity: the adjusted Rand index

When labels exist — the true generating classes in a simulation, the diseased/healthy status of a biopsy, the scaffold of a molecule — a partition can be compared with them. The **Rand index** is the fraction of object *pairs* that two partitions treat consistently (together in both, or apart in both). Its flaw is that two random partitions already agree on many pairs, so Hubert and Arabie<sup class="c"><a href="#r15">[15]</a></sup> subtracted the expectation,

\[ \mathrm{ARI} = \frac{\mathrm{RI}-\mathbb{E}[\mathrm{RI}]}{\max(\mathrm{RI})-\mathbb{E}[\mathrm{RI}]} ,\]

which is 0 for chance agreement, 1 for identity and can be slightly negative. The chunk verifies the chance baseline on our own 25-character implementation from Section 3.2, which is the only way to trust it.



In [ ]:
set.seed(0); lab3 <- kmeans(Zdesc, 3, nstart = 10)$cluster
perm <- c(3, 1, 2)[lab3]                                   # relabelled: same partition
set.seed(1); rand_lab <- sample(1:3, length(lab3), replace = TRUE)
coarse <- as.integer(lab3 == 1)
cat(sprintf("ARI(lab, itself)            = %.3f\n", ari(lab3, lab3)))
cat(sprintf("ARI(lab, relabelled lab)    = %.3f   <- invariant to label names\n", ari(lab3, perm)))
cat(sprintf("ARI(lab, random labels)     = %+.4f  <- chance\n", ari(lab3, rand_lab)))
cat(sprintf("ARI(lab, coarser 2-cluster) = %.3f\n", ari(lab3, coarse)))
set.seed(2)
cat(sprintf("ARI over 200 random label vectors: mean %+.5f\n",
            mean(replicate(200, ari(lab3, sample(1:3, length(lab3), replace = TRUE))))))



The chance baseline is confirmed: random labels average essentially zero over 200 draws and relabelling leaves the index at exactly 1. That is what makes the ARI the right currency for comparing clusterings of the same objects, as Section 8 does for the compound library.

### 7.3 Stability under resampling

The deepest test an unsupervised method can take is: *would a different sample of the same population have given the same partition?*<sup class="c"><a href="#r16">[16]</a><a href="#r17">[17]</a></sup> Draw two random 80 % subsamples, cluster each independently, compare the labellings on the objects they share, and look at the distribution over repetitions.



In [ ]:
stability_ari <- function(X, K, reps = 20, frac = 0.8, seed = 0) {
  set.seed(seed); n <- nrow(X)
  assign_to <- function(cen, Y) apply(Y, 1, function(r) which.min(colSums((t(cen) - r)^2)))
  sapply(1:reps, function(i) {
    i1 <- sample(n, floor(frac*n)); i2 <- sample(n, floor(frac*n))
    set.seed(0); m1 <- kmeans(X[i1, ], K, nstart = 5, iter.max = 50)
    set.seed(0); m2 <- kmeans(X[i2, ], K, nstart = 5, iter.max = 50)
    both <- intersect(i1, i2)
    ari(assign_to(m1$centers, X[both, ]), assign_to(m2$centers, X[both, ])) })
}
bb <- make_blobs(600, matrix(c(-6,-6, -6,6, 6,-6, 6,6), ncol = 2, byrow = TRUE), 0.6, seed = 9)
stab <- list()
for (K in c(2, 3, 4, 5, 6, 8)) stab[[sprintf("descriptors K=%d", K)]] <- stability_ari(Zdesc, K)
for (K in c(3, 4, 5))         stab[[sprintf("blobs K=%d", K)]]       <- stability_ari(bb$X, K)
for (nm in names(stab))
  cat(sprintf("%-20s stability ARI %.3f +- %.3f (min %.3f)\n",
              nm, mean(stab[[nm]]), sd(stab[[nm]]), min(stab[[nm]])))

sd_long <- do.call(rbind, lapply(names(stab), function(nm)
  data.frame(what = nm, ari = stab[[nm]],
             kind = ifelse(startsWith(nm, "desc"), "ROCK1 descriptors", "well-separated blobs"))))
sd_long$what <- factor(sd_long$what, levels = names(stab))
print(ggplot(sd_long, aes(what, ari, fill = kind)) +
        geom_boxplot(alpha = .7, outlier.size = .6) +
        scale_fill_manual(values = unname(PAL[c("blue", "green")])) +
        ylim(0, 1.02) + labs(x = NULL, y = "ARI between two 80% subsamples",
                             title = "Stability is necessary but not sufficient") +
        theme(axis.text.x = element_text(angle = 25, hjust = 1, size = 7),
              legend.position = "bottom", legend.title = element_blank()))



<p class="caption"><b>Figure 9.</b> ARI between two independent 80 % subsamples over 20 repetitions. Blue: ROCK1 descriptors; green: four well-separated synthetic blobs, where stability is essentially perfect at the correct K. The descriptor partitions at K = 2 and K = 3 are reproducible (median ARI above 0.8) even though their silhouettes in Figure 8 say the clusters are not separated, and stability collapses only once K exceeds the number of directions in which the cloud is elongated.</p>

This is the subtlest result of the unit. **A stable partition is not necessarily a real one.** Cutting a smooth, elongated cloud at its waist reproduces beautifully across subsamples, because the cloud's shape is a stable property of the population; what is unstable is the *number* of pieces, not the boundary of any one. Stability therefore refutes a clustering but cannot confirm it.

## 8. The lab's first task: finding the chemical series

A chemist looking at 2,764 ROCK1 actives asks: *how many distinct chemical ideas are here, and which compounds belong to each?* Three answers follow, each from a different notion of similarity.

**Butina/leader clustering**<sup class="c"><a href="#r18">[18]</a></sup> is the field's standard: count, for each molecule, how many neighbours lie within a cutoff \(t\); sort by that count; then walk the list, and whenever a molecule is still unclaimed make it a **cluster leader** and claim all its unclaimed neighbours. One pass, no \(K\), no pair closer than \(t\) split apart, and a real molecule at every centre. In Morgan/ECFP4 space \(t=0.35\) is the conventional "same series" cutoff. The alternatives compared here are **average linkage on the same Tanimoto matrix** and **k-means on the raw bit vectors** — chemically dubious, but what a general-purpose pipeline does by default. The external reference is the **Bemis-Murcko scaffold**<sup class="c"><a href="#r20">[20]</a></sup> already stored in the table.



In [ ]:
t0 <- proc.time()[["elapsed"]]
S <- tanimoto(FP); D <- 1 - S; diag(D) <- 0
cat(sprintf("%s Tanimoto distances in %.2f s; mean %.3f, median %.3f, 1st percentile %.3f\n",
            format(sum(upper.tri(D)), big.mark = ","), proc.time()[["elapsed"]] - t0,
            mean(D[upper.tri(D)]), median(D[upper.tri(D)]), quantile(D[upper.tri(D)], .01)))
butina <- function(D, cutoff) {
  n <- nrow(D); nb <- lapply(seq_len(n), function(i) which(D[i, ] <= cutoff))
  lab <- integer(n); cid <- 0L
  for (i in order(lengths(nb), decreasing = TRUE)) {          # most crowded molecule first
    if (lab[i] != 0L) next
    cid <- cid + 1L
    lab[c(i, setdiff(nb[[i]], which(lab != 0L)))] <- cid      # leader claims free neighbours
  }
  lab
}
but <- lapply(c(0.2, 0.3, 0.35, 0.4, 0.5, 0.6), function(cut) butina(D, cut))
names(but) <- c("0.2", "0.3", "0.35", "0.4", "0.5", "0.6")
knitr::kable(do.call(rbind, lapply(names(but), function(cut) { tt <- table(but[[cut]])
  data.frame(cutoff = as.numeric(cut), clusters = length(tt), largest = max(tt),
             singletons = sum(tt == 1), pct_in_singletons = round(100*sum(tt == 1)/nrow(D), 1)) })),
  row.names = FALSE, caption = "Butina clustering at six cutoffs")


In [ ]:
lab_butina <- but[["0.35"]]; K_but <- max(lab_butina)
lab_agg <- cutree(hclust(as.dist(D), method = "average"), K_but)
set.seed(0); lab_km <- kmeans(as.matrix(FP), K_but, nstart = 3, iter.max = 50)$cluster
lab_scaf <- as.integer(factor(cmp$scaffold_smiles))
parts <- list("Butina t=0.35" = lab_butina, "average linkage" = lab_agg,
              "k-means on bits" = lab_km, "Murcko scaffold" = lab_scaf)
for (nm in names(parts)) { tt <- table(parts[[nm]])
  cat(sprintf("%-16s clusters %5d  largest %4d  singletons %4d\n", nm, length(tt), max(tt), sum(tt == 1))) }
M <- outer(seq_along(parts), seq_along(parts),
           Vectorize(function(i, j) ari(parts[[i]], parts[[j]])))
dimnames(M) <- list(names(parts), names(parts))
knitr::kable(round(M, 3), caption = "Adjusted Rand index between four views of the same library")


In [ ]:
szdf <- do.call(rbind, lapply(c("0.2", "0.35", "0.5"), function(cut) {
  s <- sort(as.numeric(table(but[[cut]])), decreasing = TRUE)
  data.frame(rank = seq_along(s), size = s,
             cut = sprintf("t = %s (%d clusters)", cut, length(s))) }))
g1 <- ggplot(szdf, aes(rank, size, colour = cut)) + geom_line() +
  scale_x_log10() + scale_y_log10() + scale_colour_manual(values = unname(PAL)) +
  labs(x = "cluster rank", y = "cluster size", title = "Butina cluster sizes follow a power law") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
mdf <- expand.grid(a = rownames(M), b = colnames(M)); mdf$v <- as.vector(M)
g2 <- ggplot(mdf, aes(b, a, fill = v)) + geom_tile() +
  geom_text(aes(label = sprintf("%.2f", v)), size = 2.6,
            colour = ifelse(mdf$v < 0.6, "white", "black")) +
  scale_fill_gradientn(colours = c("#2b5d9b", "#6a4c93", "#d2691e", "#f2d479"), limits = c(0, 1)) +
  labs(x = NULL, y = NULL, fill = "ARI", title = "ARI between four partitions") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1, size = 7), axis.text.y = element_text(size = 7))
print(g1 | g2)



<p class="caption"><b>Figure 10.</b> Left: Butina cluster sizes on logarithmic axes for three cutoffs — heavy-tailed at every cutoff, a few large series and a long tail of singletons, the typical shape of a public bioactivity table. Right: adjusted Rand index between the four partitions; Butina and average linkage on the identical matrix agree only moderately (0.60), and both agree weakly with the formal scaffold assignment (0.28 and 0.23).</p>

Three lessons sit in that matrix.

1. **Same distances, different algorithm, different answer.** Average linkage and Butina use the *identical* matrix cut to the *identical* number of clusters, and their ARI is 0.60 — substantial disagreement about hundreds of compounds. The editions differ here: the Python edition's `scipy` average linkage chained into one enormous cluster and scored near zero against Butina, whereas `stats::hclust` gave a largest cluster of 60 and a better-behaved partition. The lesson survives in a stronger form — the agglomeration order, not only the metric, is part of the method.
2. **Clusters are not scaffolds.** An ARI of 0.28 between Butina clusters and Bemis-Murcko scaffolds means they encode related but different notions of "same series" — the scaffold ignores every substituent, the fingerprint does not. When a referee asks for a "scaffold-based split", say which of the two you used.
3. **Granularity is a choice with consequences.** Going from \(t=0.2\) to \(t=0.6\) changes the number of clusters from 1,976 to 344, nearly sixfold, and the share of compounds stranded in singletons from 57 % to 7 %. No internal criterion picks \(t\); the project does — the cutoff for buying one analogue per series is not the cutoff for a leave-cluster-out validation split<sup class="c"><a href="#r29">[29]</a></sup>.



In [ ]:
big <- which(lab_butina == as.integer(names(which.max(table(lab_butina)))))
leader <- big[which.max(sapply(big, function(i) sum(D[i, big] <= 0.35)))]
cat(sprintf("largest cluster at t = 0.35: %d compounds; distinct Murcko scaffolds inside it: %d\n",
            length(big), length(unique(cmp$scaffold_smiles[big]))))
cat(sprintf("pActivity in the cluster: median %.2f, range %.2f-%.2f (whole table: median %.2f)\n",
            median(cmp$pActivity[big]), min(cmp$pActivity[big]), max(cmp$pActivity[big]),
            median(cmp$pActivity)))
cat(sprintf("Tanimoto similarity to the leader: minimum %.3f (the cutoff guarantees >= 0.65)\n",
            min(1 - D[leader, big])))
cat("five members, by ChEMBL id and potency:\n")
print(cmp[big[1:5], .(inchikey = substr(inchikey, 1, 14), chembl_ids, pActivity,
                      scaffold = substr(scaffold_smiles, 1, 32))])



The largest series at \(t=0.35\) holds 96 compounds with a median pActivity of 7.61 against 6.44 for the table — a heavily optimised series — and spans 17 formal Murcko scaffolds: one chemical idea, seventeen "scaffolds". The minimum similarity to the leader is exactly 0.650, as the algorithm guarantees. R cannot draw the molecules here (no RDKit), so the table prints identifiers, potencies and truncated scaffold SMILES; paste any of them into ChEMBL's web interface to see the shared core.

## 9. The lab's second task: picking a diverse subset, and proving it

Suppose the budget allows 50 compounds for an orthogonal assay and the goal is **coverage**: no two picks should be near-duplicates. The right tool is not a clustering but a **dissimilarity-based picker**<sup class="c"><a href="#r21">[21]</a></sup>; the MaxMin algorithm starts from one molecule and repeatedly adds the molecule whose *minimum* distance to the already-picked set is largest. Claims of diversity are usually asserted; here we measure it against the only honest baseline, random selection of the same size, with a p-value by simulation.



In [ ]:
maxmin <- function(D, npick, seed = 0) {
  set.seed(seed); picked <- sample(nrow(D), 1); mind <- D[picked, ]
  for (k in 2:npick) {
    nxt <- which.max(mind); picked <- c(picked, nxt)
    mind <- pmin(mind, D[nxt, ]); mind[picked] <- -Inf
  }
  picked
}
NPICK <- 50
picked <- maxmin(D, NPICK)
set_d <- function(idx) D[idx, idx][upper.tri(diag(length(idx)))]
dp_ <- set_d(picked)
set.seed(3); rand_sets <- lapply(1:200, function(i) sample(nrow(D), NPICK))
mean_r <- sapply(rand_sets, function(s) mean(set_d(s)))
min_r  <- sapply(rand_sets, function(s) min(set_d(s)))
cat(sprintf("MaxMin pick : mean pairwise distance %.4f, minimum pairwise distance %.4f\n",
            mean(dp_), min(dp_)))
cat(sprintf("random picks: mean %.4f +- %.4f (200 sets), best random %.4f\n",
            mean(mean_r), sd(mean_r), max(mean_r)))
cat(sprintf("              their minimum pairwise distance averages %.4f\n", mean(min_r)))
cat(sprintf("z = %.1f standard deviations above the random mean; empirical p = %.3f\n",
            (mean(dp_) - mean(mean_r))/sd(mean_r), mean(mean_r >= mean(dp_))))
cat(sprintf("distinct scaffolds: picked %d/%d, random (mean) %.1f/%d\n",
            length(unique(cmp$scaffold_smiles[picked])), NPICK,
            mean(sapply(rand_sets, function(s) length(unique(cmp$scaffold_smiles[s])))), NPICK))


In [ ]:
g1 <- ggplot(data.frame(m = mean_r), aes(m)) +
  geom_histogram(bins = 30, fill = "grey65", colour = NA) +
  geom_vline(xintercept = mean(dp_), colour = PAL["orange"], linewidth = 1) +
  labs(x = "mean pairwise Tanimoto distance within the set", y = "sets",
       title = sprintf("Diversity vs its null (MaxMin = %.3f)", mean(dp_)))
g2 <- ggplot(data.frame(m = min_r), aes(m)) +
  geom_histogram(bins = 30, fill = "grey65", colour = NA) +
  geom_vline(xintercept = min(dp_), colour = PAL["orange"], linewidth = 1) +
  labs(x = "minimum pairwise distance within the set", y = "sets",
       title = sprintf("The redundancy a random pick buys (MaxMin = %.3f)", min(dp_)))
print(g1 | g2)



<p class="caption"><b>Figure 11.</b> Left: mean pairwise Tanimoto distance inside 200 random 50-compound subsets (grey) and inside the MaxMin pick (orange line at 0.905); the pick lies outside the entire null distribution, 8.4 standard deviations above its mean. Right: the same for the <em>minimum</em> pairwise distance — random subsets average a closest pair at 0.211, i.e. a near-twin, while the MaxMin set's closest pair is at 0.839.</p>

The mean distance moves only from about 0.84 to about 0.91, because in a 2,048-bit space almost every pair of unrelated molecules is already far apart. The second panel is where the money is: the closest pair in a random set of 50 is typically a near-twin at Tanimoto similarity 0.79, so assay wells are spent twice on one chemical idea, whereas the MaxMin set's closest pair is at similarity 0.16. **Report the minimum, not only the mean, when you claim a selection is diverse.** The scaffold count tells the same story more weakly — 50 distinct scaffolds against 45 — which is why the distance statistics belong in the methods section.

<div class="exercise"><h4>Exercise 3</h4>
<p>Replace the MaxMin pick with "one compound per Butina cluster, largest clusters first" (a <em>cluster-based</em> pick: <code>sapply(order(-table(lab_butina))[1:50], function(k) which(lab_butina == k)[1])</code>) and compute the same two statistics. Which strategy gives the higher mean distance, and which guarantees that every large series is represented? Explain why a purchasing committee might prefer the second even though it scores worse on diversity.</p></div>

## 10. The lab's third task: metastable conformations from a replica ensemble

Clustering the frames of a trajectory is how a structural project turns 500 pictures per run into "the protein visits three states, and here is a representative structure for each" — the input to a figure, a docking run or a Markov state model<sup class="c"><a href="#r23">[23]</a></sup>. The classic comparison of algorithms for trajectories is Shao and co-workers'<sup class="c"><a href="#r22">[22]</a></sup>; its practical conclusions (the metric matters more than the algorithm, and the result depends on how much of the trajectory you keep) are reproduced below. Our ensemble is deliberately awkward: six runs of only 1 ns. The coordinates are superposed on the minimised reference, so a Euclidean distance between flattened \(20\times3\) vectors divided by \(\sqrt{20}\) is the C-alpha RMSD in nanometres, and k-means can be applied directly.



In [ ]:
refv <- as.vector(t(ref_ca))
rmsd_ref <- sqrt(rowSums(sweep(Fmd, 2, refv)^2)/20)          # nm, = rmsd_ca_nm of the observables table
cat("mean C-alpha RMSD to the reference, per run (nm):\n")
print(round(tapply(rmsd_ref, run_of_frame, mean), 3))
pc_md <- prcomp(Fmd, center = TRUE, scale. = FALSE)
ev <- pc_md$sdev^2/sum(pc_md$sdev^2)
cat(sprintf("\nPCA of the 60-dimensional coordinate space: PC1 %.1f%%, PC2 %.1f%%, first two together %.1f%%\n",
            100*ev[1], 100*ev[2], 100*sum(ev[1:2])))

cond_of_frame <- substr(run_of_frame, 1, 1)
set.seed(0); mdsub <- sample(nrow(Fmd), 1500); Dmd <- dist(Fmd[mdsub, ])
mdrows <- do.call(rbind, lapply(2:8, function(K) {
  set.seed(0); kmm <- kmeans(Fmd, K, nstart = 10, iter.max = 50)
  data.frame(K = K, silhouette = mean(silhouette(kmm$cluster[mdsub], Dmd)[, 3]),
             ARI_vs_run = ari(run_of_frame, kmm$cluster),
             ARI_vs_temperature = ari(cond_of_frame, kmm$cluster),
             sizes = paste(sort(kmm$size, decreasing = TRUE), collapse = "/")) }))
knitr::kable(mdrows, digits = 3, row.names = FALSE,
             caption = "Clustering 3,000 MD frames: what do the clusters recover?")


In [ ]:
set.seed(0); kmm <- kmeans(Fmd, 4, nstart = 10, iter.max = 50); lab_md <- kmm$cluster
print(table(run = run_of_frame, cluster = lab_md))
reps <- do.call(rbind, lapply(1:4, function(c) {
  idx <- which(lab_md == c); cen <- colMeans(Fmd[idx, , drop = FALSE])
  dcen <- sqrt(rowSums(sweep(Fmd[idx, , drop = FALSE], 2, cen)^2)/20)
  med <- idx[which.min(dcen)]
  data.frame(cluster = c, frames = length(idx), run = run_of_frame[med],
             time_ps = time_of_frame[med], RMSD_medoid_to_reference_nm = round(rmsd_ref[med], 3),
             mean_RMSD_to_centroid_nm = round(mean(dcen), 3)) }))
knitr::kable(reps, row.names = FALSE, caption = "Representative (medoid) frame of each cluster")

Y <- as.data.frame(pc_md$x[, 1:2]); names(Y) <- c("PC1", "PC2")
Y$run <- run_of_frame; Y$cluster <- factor(lab_md); Y$t <- time_of_frame
g1 <- ggplot(Y, aes(PC1, PC2, colour = run)) + geom_point(size = .3, alpha = .6) +
  labs(title = "Coloured by run") + guides(colour = guide_legend(override.aes = list(size = 2))) +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6))
g2 <- ggplot(Y, aes(PC1, PC2, colour = cluster)) + geom_point(size = .3, alpha = .6) +
  scale_colour_manual(values = unname(PAL)) + labs(title = "Coloured by k-means cluster (K = 4)") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6))
g3 <- ggplot(Y, aes(t, as.integer(cluster), colour = run)) +
  geom_point(size = .2, alpha = .5, position = position_jitter(height = .12)) +
  labs(x = "time (ps)", y = "cluster index", title = "Cluster occupancy along each run") +
  theme(legend.position = "none")
print(g1 | g2 | g3)



<p class="caption"><b>Figure 12.</b> Left: the 3,000 frames on the first two principal components of the C-alpha coordinates (46.5 % and 21.3 % of the variance), coloured by run — the six runs occupy largely separate regions. Centre: the same projection coloured by the K = 4 clusters, which reproduce the run structure. Right: cluster index against time; A1, A2 and A3 each stay in one cluster for the whole nanosecond, and only B2 and B3 switch.</p>

The table quantifies what the figure shows, and it is the key result: at \(K=4\) the adjusted Rand index against **run identity** is 0.503, against **temperature** 0.058. The clusters are not metastable states of Trp-cage; they are *the six trajectories*. Each run relaxed from the same minimised structure into its own corner of shape space and stayed there (replica A3 at 0.251 nm mean RMSD, A1 at 0.110 nm, the rest between), so a frame's cluster says which simulation it came from. The cross-tabulation is worth reading row by row: cluster 3 is essentially A1 plus B1, cluster 4 is all of A3, and only B2 and B3 are split between clusters.

<div class="box bridge"><h4>Domain context — why 1 ns is not enough, and what this costs</h4>
<p>Trp-cage folds and unfolds on a timescale of microseconds; its experimental melting temperature is about 315 K. In 1 ns at 340 K the molecule has had time to jiggle, not to unfold. Worse, the frames within one run are strongly autocorrelated, so a 1 ns run contains only a handful of statistically independent samples even though it contributes 500 rows to the table. Clustering treats every row as an independent object, which is why a per-frame clustering of a short ensemble recovers the replica structure: the between-replica differences are real and large, and the within-replica variation is small and correlated. The honest sentence for a paper based on data like these is: "frames cluster by replica rather than by conformational state, indicating that 1 ns per replica does not sample the relevant transitions".</p></div>

The machinery is still useful: the medoid frames above are genuine structures (not averages, which are geometrically distorted), annotated with run and time stamp, and each cluster's mean RMSD to its own centroid — 0.057 to 0.088 nm — quantifies how tight the "state" is. That is the deliverable a structural project wants, provided the caption says "representative frame of cluster k" and not "conformational state k".

## 11. The lab's fourth task: are there patient subgroups?

The expression slice holds 138 biopsies: one diseased and one healthy gingival site from each of 69 patients<sup class="c"><a href="#r27">[27]</a></sup>. The clinical question is whether the tissues fall into molecular subtypes — a claim made for this cohort<sup class="c"><a href="#r26">[26]</a></sup>; the methodological question is what a clustering of 138 samples by 1,000 genes can support. Here we have an external label (clinical status), so every partition can be scored with the ARI rather than admired. Genes are z-scored so that highly expressed genes do not dominate, and **correlation distance** (profile shape) is compared with **Euclidean distance on z-scores**.



In [ ]:
v   <- apply(E, 1, var); sel <- order(v, decreasing = TRUE)[1:1000]
sub_expr <- E[sel, ]; samples <- colnames(sub_expr)
status <- as.integer(meta$status[match(samples, meta$sample_id)] == "diseased")
Zg <- scale(t(sub_expr))                                   # samples x genes, gene-wise z-scores
Dcorr <- as.dist(1 - cor(sub_expr))                        # correlation distance between samples
cat(sprintf("1,000 most variable genes x %d samples; selected-gene variance %.2f to %.2f\n",
            ncol(sub_expr), min(v[sel]), max(v)))
cat(sprintf("correlation distance between samples: min %.3f, median %.3f, max %.3f\n",
            min(Dcorr), median(Dcorr), max(Dcorr)))
results <- list()
for (meth in c("average", "complete", "ward.D2"))
  results[[paste0("correlation + ", meth)]] <- cutree(hclust(Dcorr, method = meth), 2)
results[["euclidean(z) + ward"]] <- cutree(hclust(dist(Zg), method = "ward.D2"), 2)
set.seed(0); results[["k-means on z-scores"]] <- kmeans(Zg, 2, nstart = 10)$cluster
pcg <- prcomp(Zg, center = FALSE)
set.seed(0); results[["k-means on 5 PCs"]] <- kmeans(pcg$x[, 1:5], 2, nstart = 10)$cluster
for (nm in names(results))
  cat(sprintf("%-22s sizes %-7s ARI vs diseased/healthy %.3f\n", nm,
              paste(table(results[[nm]]), collapse = "/"), ari(status, results[[nm]])))


In [ ]:
evg <- pcg$sdev^2/sum(pcg$sdev^2)
pdat <- data.frame(PC1 = pcg$x[, 1], PC2 = pcg$x[, 2],
                   status = factor(status, 0:1, c("healthy", "diseased")))
g1 <- ggplot(pdat, aes(PC1, PC2, colour = status)) + geom_point(size = 1.2, alpha = .85) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = sprintf("PC1 (%.0f%%)", 100*evg[1]), y = sprintf("PC2 (%.0f%%)", 100*evg[2]),
       title = "Samples in PC space") +
  theme(legend.position = "bottom", legend.title = element_blank())
print(g1)
op <- par(mfrow = c(1, 2), mar = c(3, 4, 3, 1))
for (meth in c("average", "ward.D2")) {
  hc <- hclust(Dcorr, method = meth)
  plot(hc, labels = FALSE, hang = -1, xlab = "", sub = "", ylab = "merge height",
       main = sprintf("correlation + %s\nARI = %.2f", meth, ari(status, cutree(hc, 2))), cex.main = .95)
  points(seq_along(hc$order), rep(par("usr")[3], length(hc$order)),
         col = ifelse(status[hc$order] == 1, PAL["orange"], PAL["blue"]), pch = 15, cex = .5, xpd = NA)
}
par(op)



<p class="caption"><b>Figure 13.</b> Top: the 138 biopsies on their first two principal components; diseased (orange) and healthy (blue) sites separate along PC1, which carries 34 % of the variance. Bottom: dendrograms from the <em>same</em> correlation-distance matrix under average and Ward linkage, with each leaf's clinical status drawn underneath. Average linkage chains — its two-cluster cut isolates one outlying sample and leaves the other 137 together (ARI 0.000) — while Ward splits the cohort 73/65 in line with disease status (ARI 0.384).</p>

Two methodological conclusions and one biological caution.

- **The linkage, not the distance, decided the answer.** All three correlation-based clusterings used an identical matrix; average and complete linkage both returned a 137/1 split with ARI exactly 0.000, and only Ward recovered the clinical signal. A hierarchical clustering reported without its linkage is not reproducible.
- **An ARI of 0.38 to 0.44 against a two-class label is moderate agreement, not a discovery.** The dominant axis of variation in gingival tissue is inflammation, so unsupervised clustering rediscovers the label it was not given. Plain k-means on the z-scores (0.440) beat every hierarchical variant, the opposite of what the dendrogram's visual authority suggests.
- **Subtype claims must respect the paired design.** Each patient contributed two samples, so the 138 rows are not 138 independent observations.

## 12. The clusters that are not there

The final hazard is visual. Non-linear embeddings — t-SNE<sup class="c"><a href="#r24">[24]</a><a href="#r35">[35]</a></sup> and UMAP — optimise a *neighbourhood-preserving* 2-D layout, and will produce visible groups whether or not the data contains any, because the optimisation pushes apart points that are not near neighbours. Distances between blobs, their sizes and even their number (which depends on perplexity) are not interpretable<sup class="c"><a href="#r24">[24]</a></sup>; the single-cell genomics literature has been through a public reckoning on this point<sup class="c"><a href="#r25">[25]</a></sup>.

The experiment: take 400 points of **pure 50-dimensional Gaussian noise** — a data set with, by construction, exactly one cluster — embed it with `Rtsne` at three perplexities, run k-means on the embedding, and compute the silhouette of the resulting partition twice: in the embedding, and in the original data.



In [ ]:
set.seed(1); Xnoise <- matrix(rnorm(400*50), 400, 50)
Dnoise <- dist(Xnoise); noise_emb <- list()
nrows <- do.call(rbind, lapply(c(5, 30, 100), function(perp) {
  set.seed(0)
  e <- Rtsne(Xnoise, dims = 2, perplexity = perp, pca = TRUE,
             check_duplicates = FALSE, verbose = FALSE)$Y
  set.seed(0); lab <- kmeans(e, 4, nstart = 10)$cluster
  noise_emb[[as.character(perp)]] <<- data.frame(x = e[, 1], y = e[, 2], g = factor(lab), perp = perp)
  data.frame(perplexity = perp,
             silhouette_in_embedding = mean(silhouette(lab, dist(e))[, 3]),
             silhouette_of_same_labels_in_data = mean(silhouette(lab, Dnoise)[, 3])) }))
knitr::kable(nrows, digits = 3, row.names = FALSE,
             caption = "Four k-means clusters imposed on t-SNE maps of pure noise")
cat("The data is isotropic Gaussian noise: the true number of clusters is 1.\n")

nd_all <- do.call(rbind, noise_emb)
nd_all$lab <- factor(sprintf("perplexity %d (silhouette %.2f in map, %.2f in data)", nd_all$perp,
  nrows$silhouette_in_embedding[match(nd_all$perp, nrows$perplexity)],
  nrows$silhouette_of_same_labels_in_data[match(nd_all$perp, nrows$perplexity)]))
print(ggplot(nd_all, aes(x, y, colour = g)) + geom_point(size = .7, show.legend = FALSE) +
        facet_wrap(~ lab, nrow = 1, scales = "free") +
        scale_colour_manual(values = unname(PAL)) + labs(x = NULL, y = NULL) +
        theme(axis.text = element_blank(), strip.text = element_text(size = 7)))



<p class="caption"><b>Figure 14.</b> t-SNE maps of 400 points of 50-dimensional Gaussian noise at three perplexities, coloured by k-means run on the map itself. Every panel shows apparent islands with a silhouette <em>in the map</em> of 0.33 to 0.35 — the range a referee accepts as "well separated" — while the same labels score between 0.000 and 0.015 in the original data.</p>

The rule that follows is simple and worth memorising.

> **Cluster in the data space; use the embedding only to display the clustering.** Never cluster on t-SNE or UMAP coordinates, and never count blobs in a map.

The correct practice is below: a t-SNE of the 2,764 compounds, computed from 50 truncated-SVD components of the sparse bit matrix (`irlba`, which never densifies it), coloured by the Butina clusters obtained from the full-dimensional Tanimoto distances.



In [ ]:
t0 <- proc.time()[["elapsed"]]
set.seed(0); sv <- irlba(FP, nv = 50)
P <- as.matrix(FP %*% sv$v)
set.seed(0); emb <- Rtsne(P, dims = 2, perplexity = 30, pca = FALSE,
                          check_duplicates = FALSE, verbose = FALSE)$Y
cat(sprintf("SVD + t-SNE of %d compounds in %.1f s\n", nrow(P), proc.time()[["elapsed"]] - t0))
topc <- as.integer(names(sort(table(lab_butina), decreasing = TRUE)[1:6]))
cat("six largest Butina clusters (id:size):",
    paste(topc, table(lab_butina)[as.character(topc)], sep = ":", collapse = "  "), "\n")
set.seed(0); ss <- sample(nrow(P), 1500)
cat(sprintf("silhouette of the Butina partition in the t-SNE map: %.3f; in the Tanimoto space it was computed on: %.3f\n",
            mean(silhouette(lab_butina[ss], dist(emb[ss, ]))[, 3]),
            mean(silhouette(lab_butina[ss], as.dist(D[ss, ss]))[, 3])))


In [ ]:
ed <- data.frame(x = emb[, 1], y = emb[, 2], pAct = cmp$pActivity,
                 series = ifelse(lab_butina %in% topc, as.character(lab_butina), NA))
g1 <- ggplot(ed, aes(x, y)) + geom_point(colour = "grey82", size = .4) +
  geom_point(data = subset(ed, !is.na(series)), aes(colour = series), size = .9) +
  scale_colour_manual(values = c(unname(PAL), "#8c8c3a", "#a3324f")) +
  labs(x = NULL, y = NULL, title = "Butina clusters shown on a t-SNE map") +
  theme(axis.text = element_blank(), legend.position = "bottom",
        legend.title = element_blank(), legend.text = element_text(size = 6))
g2 <- ggplot(ed, aes(x, y, colour = pAct)) + geom_point(size = .4) +
  scale_colour_viridis_c(name = "pActivity") +
  labs(x = NULL, y = NULL, title = "The same map coloured by potency") +
  theme(axis.text = element_blank(), legend.position = "bottom")
print(g1 | g2)



<p class="caption"><b>Figure 15.</b> Left: the collection embedded with t-SNE from 50 SVD components of the fingerprints; grey points are all compounds, colours mark the six largest Butina clusters (96, 56, 42, 41, 35 and 33 compounds), and each series lands in a compact island — a visual check that the clustering computed in the full space is coherent. Right: the identical layout coloured by potency, which varies within every island.</p>

Two numbers above are the inverse of the noise experiment: the Butina partition scores 0.132 in the Tanimoto space where it was computed and only 0.018 in the map, because t-SNE spends its degrees of freedom on local neighbourhoods and 961 small clusters cannot all be separated in two dimensions. A silhouette computed in an embedding is therefore uninformative in both directions: high for structure that does not exist, low for structure that does.

<div class="box bridge"><h4>Domain context — what "a cluster of actives" does and does not imply</h4>
<p>A compact island of highly potent compounds is usually a <em>publication artefact</em> before it is a biological fact: one group published a series, optimised it to nanomolar potency and deposited forty analogues, so the cluster records the history of the field rather than a privileged region of chemical space. The 96-compound series of Section 8, median pActivity 7.61 against 6.44 for the table, is such a record — the bias that makes random splits optimistic.</p></div>

<div class="box bridge"><h4>Bridge</h4>
<p>Similarity and fingerprints are developed from chemistry first principles in the cheminformatics course, units <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">U5 (descriptors and fingerprints)</a>, <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">U8 (chemical space)</a> and <a href="../../../Cheminformatics/U11_Similarity_SAR/index.html">U11 (similarity and SAR)</a>; read U5 if the word "fingerprint" is new. The simulation that produced the Trp-cage frames is explained in the bioinformatics course, <a href="../../../BGM324/05_Molecular_Dynamics/index.html">BGM324 unit 05 (molecular dynamics)</a>, and the structural-biology vocabulary in <a href="../../../BGM324/02_Protein_Structure_Prediction/index.html">unit 02</a>. The artificial-intelligence course covers unsupervised learning from the algorithms' side: <a href="../../../BGM411/R/index.html">BGM411 study material (R edition)</a>. Inside this course the natural continuations are <a href="../U16_Interpretability/index.html">Unit 16 (interpretability)</a>, which asks what a <em>supervised</em> model learned, and <a href="../U17_Uncertainty_AD/index.html">Unit 17 (uncertainty and applicability domain)</a>, which is a distance question again; the deep-learning view of learned representations is <a href="../U18_Deep_Learning/index.html">Unit 18</a>. The Python edition of this unit, with RDKit's own Butina implementation and molecule drawings, is <a href="../../Python/U15_Clustering/index.html">here</a>.</p></div>



In [ ]:
summary_tab <- data.frame(
  data = c("ROCK1 descriptors (11-D, z-scored)", "ROCK1 descriptors", "ROCK1 descriptors",
           "ROCK1 fingerprints (Tanimoto)", "ROCK1 fingerprints", "MD frames (C-alpha RMSD)",
           "138 biopsies (correlation)", "400 points of pure noise"),
  method = c(sprintf("k-means, K chosen by silhouette (K = %d)", best_sil),
             sprintf("DBSCAN by hand, eps = %.1f", dbrows$eps[3]),
             sprintf("diagonal GMM, BIC-selected (K = %d)", bst$K),
             sprintf("Butina t = 0.35 (%d clusters)", K_but),
             sprintf("average linkage, same K (%d)", K_but),
             "k-means, K = 4", "Ward linkage, cut at 2", "k-means on a t-SNE map"),
  quality = c(sprintf("silhouette %.3f", max(swp$silhouette)),
              sprintf("%.0f%% noise, %d clusters", dbrows$noise_pct[3], dbrows$clusters[3]),
              sprintf("%.0f%% ambiguous (max resp < 0.8)", 100*mean(mx < 0.8)),
              sprintf("ARI vs scaffold %.2f", ari(lab_scaf, lab_butina)),
              sprintf("ARI vs Butina %.2f", ari(lab_butina, lab_agg)),
              sprintf("ARI vs run %.2f / vs temperature %.2f", ari(run_of_frame, lab_md),
                      ari(cond_of_frame, lab_md)),
              sprintf("ARI vs status %.2f", ari(status, results[["correlation + ward.D2"]])),
              sprintf("silhouette %.2f in map vs %.2f in data",
                      nrows$silhouette_in_embedding[2], nrows$silhouette_of_same_labels_in_data[2])),
  verdict = c("a continuum, not groups", "no density scale", "BIC keeps falling", "usable series",
              "same distances, different answer", "clusters = replicas",
              "recovers the known label", "entirely artefactual"))
knitr::kable(summary_tab, row.names = FALSE)



<p class="caption"><b>Table 1.</b> Every partition computed in this unit, with the measurement that decided whether to believe it. Three of the eight rows are negative results.</p>

## 13. Summary

Clustering cannot fail loudly: it always returns labels, and the labels always look like a finding, so the discipline has to come from the analyst. This unit built that discipline in five layers.

**The metric is the model.** Euclidean distance on standardised descriptors, Tanimoto on fingerprints<sup class="c"><a href="#r19">[19]</a></sup>, RMSD on superposed conformations and correlation on expression profiles answer four different questions about closeness. The Tanimoto matrix of 2,764 compounds is three lines of `Matrix` algebra and under a second; the same matrix for a 100,000-compound catalogue would be 40 GB, so the algorithm must be chosen to avoid building it.

**Five algorithm families, each with its geometry.** k-means assumes spherical, equal-spread, equal-size clusters and failed visibly on sheared (ARI 0.48), unequal-variance (0.58), unequal-size (0.32) and non-convex (0.21) data. Agglomerative clustering returns the whole family of partitions, and the linkage is a strong assumption: on an identical matrix, average linkage isolated a single outlier from 138 biopsies while Ward recovered the clinical label. Density methods need no \(K\) and allow noise, but a single \(\varepsilon\) cannot serve several densities, and HDBSCAN<sup class="c"><a href="#r11">[11]</a></sup>, which solves this, is the one algorithm of the unit this environment could not run. Gaussian mixtures add soft assignments and BIC<sup class="c"><a href="#r14">[14]</a></sup>; a hand-written EM showed BIC falling monotonically to the largest K tried, which means the density is not a mixture of a few separated Gaussians. Butina's leader algorithm<sup class="c"><a href="#r18">[18]</a></sup> is the chemistry-specific tool: one pass, a real molecule at each centre, one interpretable cutoff.

**Validity is three measurements, not one.** Internal indices (silhouette<sup class="c"><a href="#r3">[3]</a></sup>, Davies-Bouldin<sup class="c"><a href="#r4">[4]</a></sup>, Calinski-Harabasz<sup class="c"><a href="#r5">[5]</a></sup>) presuppose convex clusters; external indices (ARI<sup class="c"><a href="#r15">[15]</a></sup>) need a label and have a verified zero at chance; stability<sup class="c"><a href="#r17">[17]</a></sup> refutes a partition but cannot confirm it, as the high stability of a low-silhouette partition showed. The elbow is not evidence — uniform noise produced the same bend, and only the gap statistic<sup class="c"><a href="#r6">[6]</a></sup> reported "no structure" correctly.

**The four lab tasks, honestly reported.** Series detection gives a usable partition — 961 clusters at \(t=0.35\), the largest holding 96 compounds across 17 formal scaffolds — provided you do not confuse clusters with scaffolds<sup class="c"><a href="#r20">[20]</a></sup>. Diversity picking can be *proved* to work against random selection (8.4 standard deviations, p < 0.005 with 200 draws), and the minimum pairwise distance is the statistic that matters. Conformational clustering of a 1 ns replica ensemble recovers the replicas (ARI 0.50 against run, 0.06 against temperature), a result about sampling rather than about the protein. Patient clustering recovers the clinical label it was not given: a sanity check, not a subtype discovery.

**The visual trap.** t-SNE will show islands in pure noise: the silhouette computed in the map was about 0.34 against 0.01 for the same labels in the data<sup class="c"><a href="#r24">[24]</a><a href="#r25">[25]</a></sup>. The converse also holds — a real Butina partition scored 0.132 in Tanimoto space and 0.018 in the map. Cluster in the data space, colour the map.

Everything here is computed from the public ROCK1 table<sup class="c"><a href="#r28">[28]</a></sup> in base R<sup class="c"><a href="#r30">[30]</a></sup> with `data.table`, the tidyverse<sup class="c"><a href="#r31">[31]</a></sup> through `ggplot2`<sup class="c"><a href="#r32">[32]</a></sup>, `cluster`<sup class="c"><a href="#r33">[33]</a></sup>, `Matrix` and `Rtsne`; `dendextend`<sup class="c"><a href="#r34">[34]</a></sup> colours dendrograms beyond what the base plot offers. What none of them provides is the decision about whether the answer means anything.

## 14. Self-assessment

<details><summary>1. Why can inertia never be used to choose K, and what does the gap statistic add?</summary>
<p>Inertia is the objective k-means minimises, and adding a centroid can only reduce it; the curve is monotone decreasing by construction, reaching zero when K = n. The gap statistic compares the observed log inertia with the log inertia of uniform reference data in the same bounding box, so it measures how much <em>more</em> structure the data has than a structureless null. In Section 7 the gap stayed within one standard error of zero for uniform noise — correctly reporting "no clusters" where the elbow of the same data suggested K = 3.</p></details>

<details><summary>2. A colleague clusters 2,000 compounds with k-means on raw 2,048-bit Morgan fingerprints and reports the centroid of each cluster as "the representative compound of the series". Give two distinct objections.</summary>
<p>First, a centroid of binary vectors is a vector of fractions: it corresponds to no molecule and cannot be drawn, synthesised or docked, whereas a leader or medoid method returns a real compound. Second, squared Euclidean distance on sparse binary data is dominated by bits zero in both molecules — 1,967 of 2,048 for the pair in Section 2.1.</p></details>

<details><summary>3. Single and average linkage on the same Tanimoto matrix give different partitions. Is one of them wrong?</summary>
<p>Neither is wrong; they encode different definitions of "close sets". Where almost all pairwise distances are large, both tend to chain, so a cut yields one giant cluster plus singletons — a correct execution of the rule and an unusable partition. Section 4 showed both returning a cluster of ~590 of 600 compounds at K = 4 while Ward returned four balanced groups. Choose the linkage before looking at results, and remember that in R <code>ward.D2</code>, not <code>ward.D</code>, implements Ward's criterion on distances.</p></details>

<details><summary>4. What does a mean silhouette of 0.29 tell you, and what does it hide?</summary>
<p>It says that on average a point is only slightly closer to its own cluster than to the nearest other cluster — a weak partition. It hides the distribution: Figure 8 showed 10–21 % of compounds with <em>negative</em> silhouettes (points closer to a neighbouring cluster than to their own) and at least one cluster whose mean is near zero. Always plot the per-point silhouettes.</p></details>

<details><summary>5. Your clustering of 138 biopsies is highly stable under 80 % resampling. Can you conclude the subgroups are real?</summary>
<p>No. Stability is necessary but not sufficient: a reproducible way of cutting a single elongated cloud in half is stable because the cloud's shape is a stable property of the population. Section 7.3 demonstrated exactly this on the descriptor data — high stability with low silhouette. Stability rules out partitions that are artefacts of one particular sample; it does not establish separation.</p></details>

<details><summary>6. Why did clustering the MD frames recover the six runs instead of conformational states, and what would you change?</summary>
<p>Each run relaxed from the same starting structure into its own region of shape space and stayed there, and consecutive frames are strongly autocorrelated. The ARI against run identity was 0.503, against temperature 0.058, and three of the six runs occupied a single cluster each for the whole nanosecond. The fixes: microsecond trajectories, more replicas, a longer discarded equilibration, and clustering in a kinetically meaningful space (time-lagged independent components) rather than raw coordinates.</p></details>

<details><summary>7. How would you choose the Butina cutoff for (a) buying one analogue per series and (b) building a leave-cluster-out validation split?</summary>
<p>(a) A tight cutoff (0.25–0.35) defines "the same chemical idea" narrowly, so one purchase per cluster still covers distinct ideas — at t = 0.35 the table gives 961 clusters. (b) A looser cutoff (0.4–0.6) produces fewer, broader clusters (344 at t = 0.6), which makes the training and test sides of the split genuinely dissimilar and the estimate appropriately pessimistic; with a tight cutoff, near-neighbours of a test compound remain in training and the split is only nominally cluster-based.</p></details>

<details><summary>8. Give the two statistics that should accompany any claim that a selected subset is diverse.</summary>
<p>The mean pairwise distance within the subset <em>and</em> the minimum pairwise distance, each compared with the distribution obtained from many random subsets of the same size. Section 9 found the MaxMin pick 8.4 standard deviations above the random mean with no random set of 200 reaching it; the minimum distance separated the two strategies far more sharply (0.839 against an average of 0.211) than the mean did (0.905 against 0.842).</p></details>

<details><summary>9. Why is Davies-Bouldin inappropriate for comparing a DBSCAN result with a k-means result?</summary>
<p>Davies-Bouldin measures the ratio of within-cluster scatter to between-centroid distance, which assumes clusters are compact around a centroid. A crescent-shaped DBSCAN cluster has a large scatter about its centroid — which may not even lie inside the cluster — so the index penalises it regardless of how well it matches the data's geometry. Internal indices are only comparable between methods that share a notion of cluster shape; otherwise compare on an external label or on a downstream task.</p></details>

<details><summary>10. A manuscript shows a UMAP of 5,000 compounds with eight visually distinct islands and concludes "eight chemotypes". What do you ask for?</summary>
<p>A clustering performed in the original fingerprint space with a stated metric and algorithm; the silhouette of that partition computed in that space; the sensitivity of the picture to perplexity and seed; and a negative control. Figure 14 showed four convincing islands in 50-dimensional Gaussian noise at every perplexity tried.</p></details>

<details><summary>11. When is it correct to say "no clusters were found"?</summary>
<p>When the evidence says so: no knee in the k-distance curve, no plateau in the eps sweep, no valleys in the OPTICS profile, a gap statistic indistinguishable from its null, low silhouettes at every K and no agreement with any external label — all of which hold for the ROCK1 descriptor space. That sentence is publishable: it tells the next student not to repeat the analysis.</p></details>

<details><summary>12. You must deliver "one representative conformation per state" to a colleague for docking. What do you hand over and why not the centroid?</summary>
<p>The medoid: the real frame whose RMSD to the cluster centroid is smallest, with its run, time stamp, cluster population and the cluster's mean RMSD to its centroid (0.057–0.088 nm here). An average of coordinates distorts bond lengths and can collapse side chains, so it cannot be docked or deposited.</p></details>

## Glossary

<dl class="gloss">
<dt>Adjusted Rand index (ARI)</dt><dd>Agreement between two partitions corrected for chance: 0 for random agreement, 1 for identity.</dd>
<dt>Butina (leader) clustering</dt><dd>Single-pass clustering in which molecules sorted by neighbour count become cluster leaders and claim all unclaimed neighbours within a distance cutoff.</dd>
<dt>Centroid / medoid</dt><dd>The mean of a cluster's members (may not be a real object) / the member closest to that mean (always a real object).</dd>
<dt>Cophenetic correlation</dt><dd>Correlation between original distances and the dendrogram heights at which pairs merge.</dd>
<dt>Core distance / reachability</dt><dd>In OPTICS, the distance to a point's minPts-th neighbour, and the larger of that and the actual distance from the point being expanded.</dd>
<dt>Davies-Bouldin index</dt><dd>Mean over clusters of the worst ratio of within-cluster scatter to between-centroid distance; lower is better.</dd>
<dt>DBSCAN</dt><dd>Density-based clustering with a radius and a minimum-point count; produces arbitrary shapes and explicit noise labels.</dd>
<dt>Elbow heuristic</dt><dd>Choosing K where the inertia curve stops falling steeply; it has no null model and bends even for structureless data.</dd>
<dt>Gap statistic</dt><dd>Difference between the expected log inertia of uniform reference data and the observed log inertia; zero means no more structure than noise.</dd>
<dt>Gaussian mixture model</dt><dd>Generative model of the data as a weighted sum of multivariate normals, fitted by expectation-maximisation, giving soft cluster memberships.</dd>
<dt>HDBSCAN</dt><dd>Hierarchical extension of DBSCAN that extracts the clusters most persistent across density levels, allowing clusters of different density.</dd>
<dt>Inertia</dt><dd>Within-cluster sum of squared distances to centroids; the k-means objective, monotone decreasing in K.</dd>
<dt>Linkage</dt><dd>Rule defining the distance between two clusters in agglomerative clustering (single, complete, average, Ward).</dd>
<dt>Metastable conformation</dt><dd>A region of conformational space a molecule lingers in, relative to the time needed to cross between regions.</dd>
<dt>Responsibility</dt><dd>In a mixture model, the posterior probability that a given component generated a given observation.</dd>
<dt>RMSD</dt><dd>Root-mean-square deviation between two superposed structures; a Euclidean distance normalised by the number of atoms.</dd>
<dt>Silhouette</dt><dd>Per-object index comparing its mean distance to its own cluster with that to the nearest other cluster; ranges from -1 to 1.</dd>
<dt>Sparse matrix</dt><dd>Storage format that keeps only the non-zero entries; <code>Matrix::sparseMatrix</code> holds the 2,764 x 2,048 fingerprint matrix in the space of its 145,000 set bits.</dd>
<dt>Stability (of a clustering)</dt><dd>Agreement, measured by ARI, between partitions obtained from independent resamples of the same data.</dd>
<dt>Tanimoto (Jaccard) coefficient</dt><dd>Shared bits divided by the union of set bits of two binary fingerprints; ignores bits absent from both.</dd>
<dt>t-SNE / UMAP</dt><dd>Non-linear embeddings that preserve local neighbourhoods for visualisation; distances and apparent clusters in the map are not quantitatively interpretable.</dd>
<dt>Ward linkage</dt><dd>Agglomerative rule that merges the pair of clusters causing the smallest increase in within-cluster sum of squares; in R, <code>ward.D2</code> for distance input.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Jain AK. Data clustering: 50 years beyond K-means. <em>Pattern Recognit Lett</em>. 2010;31(8):651-666. <a href="https://doi.org/10.1016/j.patrec.2009.09.011">doi:10.1016/j.patrec.2009.09.011</a></li>
<li id="r2">Lloyd S. Least squares quantization in PCM. <em>IEEE Trans Inf Theory</em>. 1982;28(2):129-137. <a href="https://doi.org/10.1109/TIT.1982.1056489">doi:10.1109/TIT.1982.1056489</a></li>
<li id="r3">Rousseeuw PJ. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. <em>J Comput Appl Math</em>. 1987;20:53-65. <a href="https://doi.org/10.1016/0377-0427(87)90125-7">doi:10.1016/0377-0427(87)90125-7</a></li>
<li id="r4">Davies DL, Bouldin DW. A cluster separation measure. <em>IEEE Trans Pattern Anal Mach Intell</em>. 1979;PAMI-1(2):224-227. <a href="https://doi.org/10.1109/TPAMI.1979.4766909">doi:10.1109/TPAMI.1979.4766909</a></li>
<li id="r5">Calinski T, Harabasz J. A dendrite method for cluster analysis. <em>Commun Stat</em>. 1974;3(1):1-27. <a href="https://doi.org/10.1080/03610927408827101">doi:10.1080/03610927408827101</a></li>
<li id="r6">Tibshirani R, Walther G, Hastie T. Estimating the number of clusters in a data set via the gap statistic. <em>J R Stat Soc Series B</em>. 2001;63(2):411-423. <a href="https://doi.org/10.1111/1467-9868.00293">doi:10.1111/1467-9868.00293</a></li>
<li id="r7">Ward JH. Hierarchical grouping to optimize an objective function. <em>J Am Stat Assoc</em>. 1963;58(301):236-244. <a href="https://doi.org/10.1080/01621459.1963.10500845">doi:10.1080/01621459.1963.10500845</a></li>
<li id="r8">Murtagh F, Legendre P. Ward's hierarchical agglomerative clustering method: which algorithms implement Ward's criterion? <em>J Classif</em>. 2014;31(3):274-295. <a href="https://doi.org/10.1007/s00357-014-9161-z">doi:10.1007/s00357-014-9161-z</a></li>
<li id="r9">Schubert E, Sander J, Ester M, Kriegel HP, Xu X. DBSCAN revisited, revisited: why and how you should (still) use DBSCAN. <em>ACM Trans Database Syst</em>. 2017;42(3):19. <a href="https://doi.org/10.1145/3068335">doi:10.1145/3068335</a></li>
<li id="r10">Ankerst M, Breunig MM, Kriegel HP, Sander J. OPTICS: ordering points to identify the clustering structure. <em>ACM SIGMOD Rec</em>. 1999;28(2):49-60. <a href="https://doi.org/10.1145/304181.304187">doi:10.1145/304181.304187</a></li>
<li id="r11">Campello RJGB, Moulavi D, Sander J. Density-based clustering based on hierarchical density estimates. In: <em>PAKDD 2013</em>. LNCS 7819. Springer; 2013:160-172. <a href="https://doi.org/10.1007/978-3-642-37456-2_14">doi:10.1007/978-3-642-37456-2_14</a></li>
<li id="r12">Dempster AP, Laird NM, Rubin DB. Maximum likelihood from incomplete data via the EM algorithm. <em>J R Stat Soc Series B</em>. 1977;39(1):1-22. <a href="https://doi.org/10.1111/j.2517-6161.1977.tb01600.x">doi:10.1111/j.2517-6161.1977.tb01600.x</a></li>
<li id="r13">Fraley C, Raftery AE. Model-based clustering, discriminant analysis, and density estimation. <em>J Am Stat Assoc</em>. 2002;97(458):611-631. <a href="https://doi.org/10.1198/016214502760047131">doi:10.1198/016214502760047131</a></li>
<li id="r14">Schwarz G. Estimating the dimension of a model. <em>Ann Stat</em>. 1978;6(2):461-464. <a href="https://doi.org/10.1214/aos/1176344136">doi:10.1214/aos/1176344136</a></li>
<li id="r15">Hubert L, Arabie P. Comparing partitions. <em>J Classif</em>. 1985;2(1):193-218. <a href="https://doi.org/10.1007/BF01908075">doi:10.1007/BF01908075</a></li>
<li id="r16">Monti S, Tamayo P, Mesirov J, Golub T. Consensus clustering: a resampling-based method for class discovery and visualization of gene expression microarray data. <em>Mach Learn</em>. 2003;52:91-118. <a href="https://doi.org/10.1023/A:1023949509487">doi:10.1023/A:1023949509487</a></li>
<li id="r17">Lange T, Roth V, Braun ML, Buhmann JM. Stability-based validation of clustering solutions. <em>Neural Comput</em>. 2004;16(6):1299-1323. <a href="https://doi.org/10.1162/089976604773717621">doi:10.1162/089976604773717621</a></li>
<li id="r18">Butina D. Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity: a fast and automated way to cluster small and large data sets. <em>J Chem Inf Comput Sci</em>. 1999;39(4):747-750. <a href="https://doi.org/10.1021/ci9803381">doi:10.1021/ci9803381</a></li>
<li id="r19">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r20">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r21">Snarey M, Terrett NK, Willett P, Wilton DJ. Comparison of algorithms for dissimilarity-based compound selection. <em>J Mol Graph Model</em>. 1997;15(6):372-385. <a href="https://doi.org/10.1016/S1093-3263(98)00008-4">doi:10.1016/S1093-3263(98)00008-4</a></li>
<li id="r22">Shao J, Tanner SW, Thompson N, Cheatham TE. Clustering molecular dynamics trajectories: 1. Characterizing the performance of different clustering algorithms. <em>J Chem Theory Comput</em>. 2007;3(6):2312-2334. <a href="https://doi.org/10.1021/ct700119m">doi:10.1021/ct700119m</a></li>
<li id="r23">Husic BE, Pande VS. Markov state models: from an art to a science. <em>J Am Chem Soc</em>. 2018;140(7):2386-2396. <a href="https://doi.org/10.1021/jacs.7b12191">doi:10.1021/jacs.7b12191</a></li>
<li id="r24">Wattenberg M, Viegas F, Johnson I. How to use t-SNE effectively. <em>Distill</em>. 2016;1(10):e2. <a href="https://doi.org/10.23915/distill.00002">doi:10.23915/distill.00002</a></li>
<li id="r25">Chari T, Pachter L. The specious art of single-cell genomics. <em>PLoS Comput Biol</em>. 2023;19(8):e1011288. <a href="https://doi.org/10.1371/journal.pcbi.1011288">doi:10.1371/journal.pcbi.1011288</a></li>
<li id="r26">Kebschull M, Demmer RT, Grun B, Guarnieri P, Pavlidis P, Papapanou PN. Gingival tissue transcriptomes identify distinct periodontitis phenotypes. <em>J Dent Res</em>. 2014;93(5):459-468. <a href="https://doi.org/10.1177/0022034514527288">doi:10.1177/0022034514527288</a></li>
<li id="r27">Papapanou PN, Behle JH, Kebschull M, et al. Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <em>BMC Microbiol</em>. 2009;9:221. <a href="https://doi.org/10.1186/1471-2180-9-221">doi:10.1186/1471-2180-9-221</a></li>
<li id="r28">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r29">Kramer C, Gedeck P. Leave-cluster-out cross-validation is appropriate for scoring functions derived from diverse protein data sets. <em>J Chem Inf Model</em>. 2010;50(11):1961-1969. <a href="https://doi.org/10.1021/ci100264e">doi:10.1021/ci100264e</a></li>
<li id="r30">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r31">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r32">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r33">Kaufman L, Rousseeuw PJ. <em>Finding Groups in Data: An Introduction to Cluster Analysis</em>. Wiley; 1990. <a href="https://doi.org/10.1002/9780470316801">doi:10.1002/9780470316801</a></li>
<li id="r34">Galili T. dendextend: an R package for visualizing, adjusting and comparing trees of hierarchical clustering. <em>Bioinformatics</em>. 2015;31(22):3718-3720. <a href="https://doi.org/10.1093/bioinformatics/btv428">doi:10.1093/bioinformatics/btv428</a></li>
<li id="r35">van der Maaten L. Accelerating t-SNE using tree-based algorithms. <em>J Mach Learn Res</em>. 2014;15:3221-3245. <a href="https://doi.org/10.48550/arXiv.1301.3342">doi:10.48550/arXiv.1301.3342</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. - Section 14.3: k-means, k-medoids, hierarchical clustering, the gap statistic.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. - Chapter 12, with R laboratories for k-means and hierarchical clustering.</li>
<li>Wickham H, Cetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023 (free online). - The data-handling idioms used throughout this edition.</li>
<li>Leach AR, Gillet VJ. <em>An Introduction to Chemoinformatics</em>. Rev. ed. Springer; 2007. - Similarity, clustering and diversity selection from the chemistry side.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>CRAN Task View "Cluster Analysis &amp; Finite Mixture Models": <a href="https://cran.r-project.org/view=Cluster">cran.r-project.org/view=Cluster</a>.</li>
<li><code>cluster</code> package reference manual (silhouette, pam, agnes, diana): <a href="https://cran.r-project.org/package=cluster">cran.r-project.org/package=cluster</a>.</li>
<li><code>dbscan</code> package (DBSCAN, OPTICS, HDBSCAN in C++; not installed in this course environment): <a href="https://cran.r-project.org/package=dbscan">cran.r-project.org/package=dbscan</a>.</li>
<li><code>mclust</code> package, model-based clustering with the full covariance family: <a href="https://cran.r-project.org/package=mclust">cran.r-project.org/package=mclust</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts on clustering and diversity selection: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
